# Reference-Based Behavioural Unusualness Analysis

**Purpose:** "For a high-risk Class-3 transaction, which of its behavioural characteristics are
actually unusual compared with an appropriate labelled reference population?"

This is **not** another ML experiment, not retraining, not model tuning, and it does not change any
risk score or the 0.90 threshold. All model, preprocessing, inference, forensic-profile, and
attribution artifacts from Notebooks 05–07 are **frozen** and reused unchanged.

This notebook contributes a distinct kind of evidence from Notebook 07:

- **Notebook 07 = model sensitivity** ("perturbing FIN6 changed the illicit logit by X").
- **Notebook 08 = reference unusualness** ("FIN6 is at the 99th percentile of the labelled
  reference population").

These are never conflated into a single number here — see Section 10.

## 0. Imports and paths

In [1]:
import time, os, json
import numpy as np
import pandas as pd
from scipy import stats

t0 = time.time()
def log(msg): print(f"[{time.time()-t0:7.1f}s] {msg}", flush=True)

DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"
PROCESSED_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\processed"
ARTIFACT_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42"
OUTPUTS_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs"
print("Paths configured.")

Paths configured.


## 1. Load frozen inputs

Loads the already-produced Class-3 scores, top-20 cases, top-20 raw profiles, and the Notebook 07
attribution tables. None of these are regenerated. The 8 representative cases are reused exactly
as selected in Notebook 07 — no new case selection occurs here.

In [2]:
print("="*70); print("SECTION 1: LOAD FROZEN INPUTS"); print("="*70)

risk_scores = pd.read_csv(os.path.join(OUTPUTS_DIR, "unknown_transaction_risk_scores.csv"))
top20_scores = pd.read_csv(os.path.join(OUTPUTS_DIR, "top_unknown_forensic_cases.csv"))
top20_profiles = pd.read_csv(os.path.join(OUTPUTS_DIR, "top20_forensic_case_profiles.csv"))
attributions = pd.read_csv(os.path.join(OUTPUTS_DIR, "top_forensic_model_attributions.csv"))
attribution_summary = pd.read_csv(os.path.join(OUTPUTS_DIR, "forensic_case_attribution_summary.csv"))
assert len(risk_scores) == 157205
assert len(top20_scores) == 20
print(f"Loaded frozen Class-3 scores ({len(risk_scores)} rows), top-20 cases, top-20 profiles, "
      f"attributions ({len(attributions)} rows, {attributions['txId'].nunique()} cases), attribution summary.")

representative_txids = attribution_summary["txId"].tolist()
print(f"Representative cases (from Notebook 07, reused unchanged): {representative_txids}")

SECTION 1: LOAD FROZEN INPUTS


Loaded frozen Class-3 scores (157205 rows), top-20 cases, top-20 profiles, attributions (1488 rows, 8 cases), attribution summary.
Representative cases (from Notebook 07, reused unchanged): [99942236, 99942297, 99942037, 99942047, 99942248, 196098540, 15340246, 191501490]


## 2. Rebuild raw (pre-scaling) B-representation feature values

Same construction as Notebooks 05–07 (same raw files, same node ordering, same merge), used here
only to retrieve human-readable raw feature values for descriptive statistics — no scaling, no
model, no preprocessing objects are needed in this notebook.

In [3]:
print("\n" + "="*70); print("SECTION 2: REBUILD RAW B-REPRESENTATION VALUES"); print("="*70)

with open(os.path.join(ARTIFACT_DIR, "feature_columns.json")) as f:
    saved_feature_cols = json.load(f)

tx_features = pd.read_csv(DATA_DIR + r"\txs_features.csv")
tx_classes = pd.read_csv(DATA_DIR + r"\txs_classes.csv")
tx_data = tx_features.merge(tx_classes, on="txId", how="left")
tx_data = tx_data.sort_values("txId", kind="mergesort").reset_index(drop=True)
assert len(tx_data) == 203769

model_feature_cols = [c for c in tx_features.columns if c != "txId"]
REMOVE_13 = [
    "Local_feature_49", "Local_feature_65", "Local_feature_30", "Local_feature_29",
    "Local_feature_23", "Local_feature_47", "Local_feature_25", "Local_feature_43",
    "Local_feature_41", "Local_feature_54", "Local_feature_53", "Local_feature_55",
    "Local_feature_18",
]
a2_cols = [c for c in model_feature_cols if c not in REMOVE_13]
augmented_features = [
    "in_txs_degree", "out_txs_degree", "total_BTC", "fees", "size",
    "num_input_addresses", "num_output_addresses",
    "in_BTC_min", "in_BTC_max", "in_BTC_mean", "in_BTC_median", "in_BTC_total",
    "out_BTC_min", "out_BTC_max", "out_BTC_mean", "out_BTC_median", "out_BTC_total",
]
NEW_14 = ["FIN3","FIN4","FIN5","FIN6","STR1","STR2","TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","NET1","NET2"]

new_feats = pd.read_parquet(os.path.join(PROCESSED_DIR, "transaction_time_safe_features_v1.parquet"))
merged = tx_data[["txId", "class"] + a2_cols].merge(
    new_feats[["txId"] + NEW_14 + ["TMP6_is_first_appearance"]], on="txId", how="left"
)
assert len(merged) == len(tx_data)
merged["augmented_features_missing"] = merged[augmented_features].isna().all(axis=1).astype(int)
final_feature_cols_b = a2_cols + ["augmented_features_missing"] + NEW_14 + ["TMP6_is_first_appearance"]
assert final_feature_cols_b == saved_feature_cols, "STOP: feature ordering mismatch!"
print(f"[PASS] Raw feature construction matches feature_columns.json exactly ({len(final_feature_cols_b)} features).")


SECTION 2: REBUILD RAW B-REPRESENTATION VALUES
[PASS] Raw feature construction matches feature_columns.json exactly (186 features).


## 3. Reference populations

**A. Overall labelled reference population** = Class 1 + Class 2, a simple union (not reweighted,
not resampled). **B. Class-specific reference populations** = Class 1 (labelled illicit) and Class
2 (labelled licit), kept separate. **Class 3 is never used to define any reference distribution.**

In [4]:
print("\n" + "="*70); print("SECTION 3: REFERENCE POPULATIONS"); print("="*70)

labelled = merged[merged["class"].isin([1, 2])].copy()
class1_illicit = labelled[labelled["class"] == 1].copy()
class2_licit = labelled[labelled["class"] == 2].copy()
print(f"A. Overall labelled reference population (Class 1 + Class 2, simple union, NOT reweighted): n={len(labelled)}")
print(f"B. Class-specific reference populations: Class 1 (labelled illicit) n={len(class1_illicit)}, "
      f"Class 2 (labelled licit) n={len(class2_licit)}")
print("Class 3 (unknown) transactions are NEVER used to define any reference distribution.")


SECTION 3: REFERENCE POPULATIONS
A. Overall labelled reference population (Class 1 + Class 2, simple union, NOT reweighted): n=46564
B. Class-specific reference populations: Class 1 (labelled illicit) n=4545, Class 2 (labelled licit) n=42019
Class 3 (unknown) transactions are NEVER used to define any reference distribution.


## 4. Features analyzed

Only features with a **verified, documented meaning** in this project are analyzed: the 14
transaction-time-safe engineered features, the `TMP6_is_first_appearance` indicator, and the named
original/augmented transaction-level quantities (financial totals, address counts, degrees, `Time
step`). No `Local_feature_*` inherited column is included — this project's temporal-availability
audit (`03_Feature_Temporal_Audit.ipynb`) found the inherited features' semantic meanings
incompletely verified, so no meaning is invented for them here.

In [5]:
print("\n" + "="*70); print("SECTION 4: FEATURES ANALYZED"); print("="*70)

FINANCIAL = ["FIN3","FIN4","FIN5","FIN6","total_BTC","fees","size",
             "in_BTC_min","in_BTC_max","in_BTC_mean","in_BTC_median","in_BTC_total",
             "out_BTC_min","out_BTC_max","out_BTC_mean","out_BTC_median","out_BTC_total"]
STRUCTURAL = ["num_input_addresses","num_output_addresses","STR1","STR2","in_txs_degree","out_txs_degree"]
TEMPORAL = ["TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance","Time step"]
NETWORK = ["NET1","NET2"]
ALL_FEATURES = FINANCIAL + STRUCTURAL + TEMPORAL + NETWORK
BINARY_FEATURES = {"STR2", "TMP6_is_first_appearance"}
CONTINUOUS_FEATURES = [f for f in ALL_FEATURES if f not in BINARY_FEATURES]

def categorize(feat):
    if feat in FINANCIAL: return "FINANCIAL"
    if feat in STRUCTURAL: return "STRUCTURAL"
    if feat in TEMPORAL: return "TEMPORAL"
    if feat in NETWORK: return "NETWORK"
    return "OTHER"

print(f"Total features analyzed: {len(ALL_FEATURES)} (Financial={len(FINANCIAL)}, Structural={len(STRUCTURAL)}, "
      f"Temporal={len(TEMPORAL)}, Network={len(NETWORK)})")
print(f"Binary/categorical features (frequency/proportion only): {sorted(BINARY_FEATURES)}")
print("No inherited Local_feature_* columns are included: their semantic meanings have not been")
print("verified in this project's feature dictionary (03_Feature_Temporal_Audit.ipynb found the")
print("inherited features' meanings incompletely verified), so no meaning is invented for them here.")


SECTION 4: FEATURES ANALYZED
Total features analyzed: 33 (Financial=17, Structural=6, Temporal=8, Network=2)
Binary/categorical features (frequency/proportion only): ['STR2', 'TMP6_is_first_appearance']
No inherited Local_feature_* columns are included: their semantic meanings have not been
verified in this project's feature dictionary (03_Feature_Temporal_Audit.ipynb found the
inherited features' meanings incompletely verified), so no meaning is invented for them here.


## 5. Reference statistics

Continuous features: count, mean, median, std, min, max, p25/p75/p90/p95/p99, computed separately
for Overall Labelled, Class 1, and Class 2. Binary features: frequency and proportion only — no
mean/percentile statistics are computed for a 2-valued flag.

In [6]:
print("\n" + "="*70); print("SECTION 5: REFERENCE STATISTICS"); print("="*70)

def continuous_stats(series):
    s = series.dropna()
    if len(s) == 0:
        return {k: np.nan for k in ["count","mean","median","std","min","max","p25","p75","p90","p95","p99"]}
    return {
        "count": int(len(s)), "mean": float(s.mean()), "median": float(s.median()), "std": float(s.std()),
        "min": float(s.min()), "max": float(s.max()),
        "p25": float(s.quantile(0.25)), "p75": float(s.quantile(0.75)),
        "p90": float(s.quantile(0.90)), "p95": float(s.quantile(0.95)), "p99": float(s.quantile(0.99)),
    }

populations = {"Overall_Labelled": labelled, "Class1_Illicit": class1_illicit, "Class2_Licit": class2_licit}

cont_stat_rows = []
for feat in CONTINUOUS_FEATURES:
    for pop_name, pop_df in populations.items():
        st = continuous_stats(pop_df[feat])
        cont_stat_rows.append({"feature": feat, "feature_category": categorize(feat), "population": pop_name, **st})
cont_stats_df = pd.DataFrame(cont_stat_rows)
cont_stats_path = os.path.join(OUTPUTS_DIR, "reference_population_statistics.csv")
cont_stats_df.to_csv(cont_stats_path, index=False)
print(f"Saved: {cont_stats_path}  ({len(cont_stats_df)} rows: {len(CONTINUOUS_FEATURES)} features x 3 populations)")

bin_stat_rows = []
for feat in sorted(BINARY_FEATURES):
    for pop_name, pop_df in populations.items():
        s = pop_df[feat].dropna()
        count_total = int(len(s))
        count_1 = int((s == 1).sum())
        proportion_1 = count_1 / count_total if count_total else np.nan
        bin_stat_rows.append({"feature": feat, "feature_category": categorize(feat), "population": pop_name,
                               "count_total": count_total, "count_1": count_1, "proportion_1": proportion_1})
bin_stats_df = pd.DataFrame(bin_stat_rows)
bin_stats_path = os.path.join(OUTPUTS_DIR, "reference_binary_statistics.csv")
bin_stats_df.to_csv(bin_stats_path, index=False)
print(f"Saved: {bin_stats_path}  ({len(bin_stats_df)} rows: {len(BINARY_FEATURES)} features x 3 populations)")
print("\nExample continuous stats (FIN6, fee rate):")
print(cont_stats_df[cont_stats_df["feature"]=="FIN6"].to_string(index=False))
print("\nExample binary stats (TMP6_is_first_appearance):")
print(bin_stats_df[bin_stats_df["feature"]=="TMP6_is_first_appearance"].to_string(index=False))


SECTION 5: REFERENCE STATISTICS
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\reference_population_statistics.csv  (93 rows: 31 features x 3 populations)
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\reference_binary_statistics.csv  (6 rows: 2 features x 3 populations)

Example continuous stats (FIN6, fee rate):
feature feature_category       population  count     mean       median      std          min      max          p25      p75      p90      p95      p99
   FIN6        FINANCIAL Overall_Labelled  46045 0.000002 1.031204e-06 0.000007 0.000000e+00 0.000176 4.886948e-07 0.000002 0.000004 0.000004 0.000007
   FIN6        FINANCIAL   Class1_Illicit   4545 0.000002 1.054167e-06 0.000001 6.024096e-08 0.000008 1.006911e-06 0.000002 0.000004 0.000005 0.000007
   FIN6        FINANCIAL     Class2_Licit  41500 0.000002 8.983726e-07 0.000008 0.000000e+00 0.000176 4.444445e-07 0.000002 0.000004 0.000004 0.000006

Example binary stats (TMP6_is_firs

## 6. Percentile + class-specific comparison — 8 representative cases

For every representative Class-3 transaction and every analyzable feature, its **empirical
percentile relative to the overall labelled population** is computed, alongside the Class-1 and
Class-2 medians for context. Unusualness is defined explicitly: `HIGH` if percentile ≥ 95,
`LOW` if percentile ≤ 5, otherwise `NOT_EXTREME`. Binary features report reference prevalence
instead of a percentile.

In [7]:
print("\n" + "="*70); print("SECTION 6: PERCENTILE ANALYSIS -- 8 REPRESENTATIVE CASES"); print("="*70)

merged_indexed = merged.set_index("txId")

def percentile_of(value, reference_series):
    ref = reference_series.dropna().values
    if pd.isna(value) or len(ref) == 0:
        return None
    return float(stats.percentileofscore(ref, value, kind="mean"))

def unusualness_flag(pct):
    if pct is None:
        return "N/A"
    if pct >= 95: return "HIGH"
    if pct <= 5: return "LOW"
    return "NOT_EXTREME"

def build_case_rows(txid_list, score_lookup):
    rows = []
    for txid in txid_list:
        if txid not in merged_indexed.index:
            continue
        rec = merged_indexed.loc[txid]
        score = score_lookup.get(txid, np.nan)
        for feat in ALL_FEATURES:
            val = rec[feat]
            if feat in BINARY_FEATURES:
                rows.append({
                    "txId": txid, "Time step": rec["Time step"], "illicit_risk_score": score,
                    "feature": feat, "feature_category": categorize(feat), "value_type": "binary",
                    "value": val,
                    "reference_prevalence_overall": bin_stats_df.query("feature==@feat and population=='Overall_Labelled'")["proportion_1"].iloc[0],
                    "reference_prevalence_class1": bin_stats_df.query("feature==@feat and population=='Class1_Illicit'")["proportion_1"].iloc[0],
                    "reference_prevalence_class2": bin_stats_df.query("feature==@feat and population=='Class2_Licit'")["proportion_1"].iloc[0],
                    "percentile_overall_labelled": np.nan, "unusualness": "N/A (binary -- see reference_prevalence)",
                    "class1_median": np.nan, "class2_median": np.nan, "overall_median": np.nan,
                })
            else:
                pct = percentile_of(val, labelled[feat])
                c1_med = cont_stats_df.query("feature==@feat and population=='Class1_Illicit'")["median"].iloc[0]
                c2_med = cont_stats_df.query("feature==@feat and population=='Class2_Licit'")["median"].iloc[0]
                ov_med = cont_stats_df.query("feature==@feat and population=='Overall_Labelled'")["median"].iloc[0]
                rows.append({
                    "txId": txid, "Time step": rec["Time step"], "illicit_risk_score": score,
                    "feature": feat, "feature_category": categorize(feat), "value_type": "continuous",
                    "value": val,
                    "reference_prevalence_overall": np.nan, "reference_prevalence_class1": np.nan, "reference_prevalence_class2": np.nan,
                    "percentile_overall_labelled": pct, "unusualness": unusualness_flag(pct),
                    "class1_median": c1_med, "class2_median": c2_med, "overall_median": ov_med,
                })
    return pd.DataFrame(rows)

score_lookup_top20 = top20_scores.set_index("txId")["illicit_risk_score"].to_dict()

case_analysis_df = build_case_rows(representative_txids, score_lookup_top20)
case_analysis_path = os.path.join(OUTPUTS_DIR, "case_feature_percentile_analysis.csv")
case_analysis_df.to_csv(case_analysis_path, index=False)
print(f"Saved: {case_analysis_path}  ({len(case_analysis_df)} rows: {len(representative_txids)} cases x {len(ALL_FEATURES)} features)")

print("\nExample -- txId", representative_txids[0], "FIN6 and num_input_addresses:")
ex = case_analysis_df[(case_analysis_df["txId"]==representative_txids[0]) & (case_analysis_df["feature"].isin(["FIN6","num_input_addresses"]))]
print(ex[["feature","value","percentile_overall_labelled","class1_median","class2_median","unusualness"]].to_string(index=False))

n_high = int((case_analysis_df["unusualness"]=="HIGH").sum())
n_low = int((case_analysis_df["unusualness"]=="LOW").sum())
n_not_extreme = int((case_analysis_df["unusualness"]=="NOT_EXTREME").sum())
print(f"\nAcross {len(representative_txids)} representative cases x {len(CONTINUOUS_FEATURES)} continuous features "
      f"({len(representative_txids)*len(CONTINUOUS_FEATURES)} value-checks): HIGH={n_high}  LOW={n_low}  NOT_EXTREME={n_not_extreme}")


SECTION 6: PERCENTILE ANALYSIS -- 8 REPRESENTATIVE CASES
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\case_feature_percentile_analysis.csv  (264 rows: 8 cases x 33 features)

Example -- txId 99942236 FIN6 and num_input_addresses:
            feature     value  percentile_overall_labelled  class1_median  class2_median unusualness
               FIN6  0.000006                    98.668694       0.000001   8.983726e-07        HIGH
num_input_addresses       1.0                    26.715170       1.000000   2.000000e+00 NOT_EXTREME

Across 8 representative cases x 31 continuous features (248 value-checks): HIGH=19  LOW=11  NOT_EXTREME=210


## 7. Batch descriptive analysis — top 20 cases

Same schema applied to all 20 top candidates (not just the 8 representative cases), to see which
features are *most often* flagged unusual across the broader candidate set.

In [8]:
print("\n" + "="*70); print("SECTION 7: BATCH PERCENTILE ANALYSIS -- TOP 20 CASES"); print("="*70)

top20_txids = top20_scores["txId"].tolist()
top20_analysis_df = build_case_rows(top20_txids, score_lookup_top20)
top20_analysis_path = os.path.join(OUTPUTS_DIR, "top20_batch_percentile_analysis.csv")
top20_analysis_df.to_csv(top20_analysis_path, index=False)
print(f"Saved: {top20_analysis_path}  ({len(top20_analysis_df)} rows: 20 cases x {len(ALL_FEATURES)} features)")

freq_unusual = (top20_analysis_df[top20_analysis_df["value_type"]=="continuous"]
                .groupby("feature")["unusualness"]
                .apply(lambda s: (s=="HIGH").sum() + (s=="LOW").sum())
                .sort_values(ascending=False))
print("\nFeatures most often flagged HIGH or LOW unusualness across top-20 cases (count out of 20):")
print(freq_unusual.head(10).to_string())


SECTION 7: BATCH PERCENTILE ANALYSIS -- TOP 20 CASES
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top20_batch_percentile_analysis.csv  (660 rows: 20 cases x 33 features)

Features most often flagged HIGH or LOW unusualness across top-20 cases (count out of 20):
feature
FIN6             14
TMP5             14
size              8
FIN5              3
TMP2              3
in_BTC_total      3
total_BTC         2
Time step         2
out_BTC_total     2
out_BTC_max       2


## 8. Multi-feature behavioural fingerprint (independently verified)

Directly recomputes, from the raw data (not reused from memory), how often the combination
"1 input address AND 1 output address AND first-ever appearance" occurs in each population,
including the top-20 Class-3 candidates.

In [9]:
print("\n" + "="*70); print("SECTION 8: 1-IN / 1-OUT / FIRST-APPEARANCE FINGERPRINT"); print("="*70)

def fingerprint_mask(df):
    return (df["num_input_addresses"] == 1) & (df["num_output_addresses"] == 1) & (df["TMP6_is_first_appearance"] == 1)

fp_rows = []
for pop_name, pop_df in populations.items():
    m = fingerprint_mask(pop_df)
    fp_rows.append({"population": pop_name, "count": int(m.sum()), "total": int(len(pop_df)), "percentage": float(m.mean()*100)})

top20_merged_raw = merged_indexed.loc[top20_txids].reset_index()
m_top20 = fingerprint_mask(top20_merged_raw)
fp_rows.append({"population": "Top20_Class3", "count": int(m_top20.sum()), "total": int(len(top20_merged_raw)), "percentage": float(m_top20.mean()*100)})

fingerprint_df = pd.DataFrame(fp_rows)
fingerprint_path = os.path.join(OUTPUTS_DIR, "fingerprint_reference_analysis.csv")
fingerprint_df.to_csv(fingerprint_path, index=False)
print(fingerprint_df.round(4).to_string(index=False))
print(f"\nSaved: {fingerprint_path}")
print(f"\nFinding: the combination is present in {fp_rows[-1]['percentage']:.2f}% of the top-20 Class-3 candidates "
      f"compared with {fp_rows[0]['percentage']:.2f}% of the overall labelled reference population "
      f"({fp_rows[1]['percentage']:.2f}% of labelled illicit, {fp_rows[2]['percentage']:.2f}% of labelled licit). "
      f"This is reported as a description of relative rarity, not as proof of illicit activity.")


SECTION 8: 1-IN / 1-OUT / FIRST-APPEARANCE FINGERPRINT
      population  count  total  percentage
Overall_Labelled   3694  46564      7.9332
  Class1_Illicit   2897   4545     63.7404
    Class2_Licit    797  42019      1.8968
    Top20_Class3     20     20    100.0000

Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\fingerprint_reference_analysis.csv

Finding: the combination is present in 100.00% of the top-20 Class-3 candidates compared with 7.93% of the overall labelled reference population (63.74% of labelled illicit, 1.90% of labelled licit). This is reported as a description of relative rarity, not as proof of illicit activity.


## 9. Separating four distinct concepts

A feature can be rare without being illicit. This section reports, **separately, for each of a
small set of key features**: (A) rarity/unusualness in the overall labelled population, (B)
association with the labelled Class-1 (illicit) vs. Class-2 (licit) subpopulations, (C) model
sensitivity (mean |attribution| from Notebook 07), and (D) forensic interpretation — which is
explicitly left to human review, not automated by this notebook.

In [10]:
print("\n" + "="*70); print("SECTION 9: SEPARATING FOUR DISTINCT CONCEPTS"); print("="*70)

concept_rows = []
key_features = ["num_input_addresses", "num_output_addresses", "TMP6_is_first_appearance", "FIN6", "FIN3", "TMP5"]
for feat in key_features:
    if feat in BINARY_FEATURES:
        overall_prev = bin_stats_df.query("feature==@feat and population=='Overall_Labelled'")["proportion_1"].iloc[0]
        c1_prev = bin_stats_df.query("feature==@feat and population=='Class1_Illicit'")["proportion_1"].iloc[0]
        c2_prev = bin_stats_df.query("feature==@feat and population=='Class2_Licit'")["proportion_1"].iloc[0]
        rarity = f"{overall_prev*100:.2f}% prevalence overall"
        class1_assoc = f"Class1={c1_prev*100:.2f}% vs Class2={c2_prev*100:.2f}%"
    else:
        overall_median = cont_stats_df.query("feature==@feat and population=='Overall_Labelled'")["median"].iloc[0]
        c1_median = cont_stats_df.query("feature==@feat and population=='Class1_Illicit'")["median"].iloc[0]
        c2_median = cont_stats_df.query("feature==@feat and population=='Class2_Licit'")["median"].iloc[0]
        rarity = f"overall median={overall_median:.4g}"
        class1_assoc = f"Class1 median={c1_median:.4g} vs Class2 median={c2_median:.4g}"
    mean_abs_attr = attributions[attributions["feature"]==feat]["absolute_attribution"].mean() if feat in attributions["feature"].values else np.nan
    concept_rows.append({
        "feature": feat,
        "A_rarity_unusualness": rarity,
        "B_class1_association": class1_assoc,
        "C_model_sensitivity_mean_abs_attribution": mean_abs_attr,
        "D_forensic_interpretation": "Human review required -- not automated by this notebook",
    })
concept_df = pd.DataFrame(concept_rows)
print(concept_df.to_string(index=False))


SECTION 9: SEPARATING FOUR DISTINCT CONCEPTS
                 feature      A_rarity_unusualness                               B_class1_association  C_model_sensitivity_mean_abs_attribution                               D_forensic_interpretation
     num_input_addresses          overall median=1                 Class1 median=1 vs Class2 median=2                                  0.189166 Human review required -- not automated by this notebook
    num_output_addresses          overall median=2                 Class1 median=1 vs Class2 median=2                                  0.061084 Human review required -- not automated by this notebook
TMP6_is_first_appearance 61.71% prevalence overall                     Class1=89.57% vs Class2=58.69%                                  0.000000 Human review required -- not automated by this notebook
                    FIN6  overall median=1.031e-06 Class1 median=1.054e-06 vs Class2 median=8.984e-07                                  0.284790 Human revi

## 10. Combined interpretation framework (Notebook 07 + Notebook 08, not conflated)

```
                 Feature
                    |
          +---------+---------+
          v                   v
Reference unusualness     Model sensitivity
   (this notebook)        (Notebook 07)
          |                   |
          +---------+---------+
                    v
       Supporting behavioural
             indicator
```

For each representative case, the most reference-unusual features (this notebook) and the most
model-attributed features (Notebook 07) are placed **side by side per category**, never merged
into one score. Agreement between the two is informative; disagreement is equally informative and
is not hidden.

In [11]:
print("\n" + "="*70); print("SECTION 10: COMBINED INTERPRETATION FRAMEWORK"); print("="*70)

def most_unusual_features_str(txid, category, k=3):
    sub = case_analysis_df[(case_analysis_df["txId"]==txid) & (case_analysis_df["feature_category"]==category)
                            & (case_analysis_df["value_type"]=="continuous")].copy()
    sub["extremity"] = (sub["percentile_overall_labelled"] - 50).abs()
    sub = sub.sort_values("extremity", ascending=False).head(k)
    return "; ".join(f"{r['feature']}(pct={r['percentile_overall_labelled']:.1f},{r['unusualness']})" for _, r in sub.iterrows())

combined_rows = []
attr_summary_indexed = attribution_summary.set_index("txId")
for txid in representative_txids:
    a = attr_summary_indexed.loc[txid]
    combined_rows.append({
        "txId": txid, "Time step": a["Time step"], "illicit_risk_score": a["illicit_risk_score"], "logit_margin": a["logit_margin"],
        "most_unusual_financial (Notebook 08)": most_unusual_features_str(txid, "FINANCIAL"),
        "most_attributed_financial (Notebook 07)": a["top_financial_features"],
        "most_unusual_structural (Notebook 08)": most_unusual_features_str(txid, "STRUCTURAL"),
        "most_attributed_structural (Notebook 07)": a["top_structural_features"],
        "most_unusual_temporal (Notebook 08)": most_unusual_features_str(txid, "TEMPORAL"),
        "most_attributed_temporal (Notebook 07)": a["top_temporal_features"],
        "most_unusual_network (Notebook 08)": most_unusual_features_str(txid, "NETWORK"),
        "most_attributed_network (Notebook 07)": a["top_network_features"],
    })
combined_df = pd.DataFrame(combined_rows)
combined_path = os.path.join(OUTPUTS_DIR, "combined_case_interpretation_summary.csv")
combined_df.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}  ({len(combined_df)} rows)")
print("\nExample (first representative case):")
for col in combined_df.columns:
    print(f"  {col}: {combined_df.iloc[0][col]}")


SECTION 10: COMBINED INTERPRETATION FRAMEWORK
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\combined_case_interpretation_summary.csv  (8 rows)

Example (first representative case):
  txId: 99942236
  Time step: 42
  illicit_risk_score: 1.0
  logit_margin: 21.1152286529541
  most_unusual_financial (Notebook 08): FIN6(pct=98.7,HIGH); FIN5(pct=94.0,NOT_EXTREME); FIN4(pct=93.8,NOT_EXTREME)
  most_attributed_financial (Notebook 07): FIN6(-0.279); FIN3(-0.135); size(-0.113)
  most_unusual_structural (Notebook 08): num_output_addresses(pct=6.2,NOT_EXTREME); out_txs_degree(pct=12.5,NOT_EXTREME); num_input_addresses(pct=26.7,NOT_EXTREME)
  most_attributed_structural (Notebook 07): num_input_addresses(-0.192); out_txs_degree(-0.073); num_output_addresses(-0.065)
  most_unusual_temporal (Notebook 08): TMP5(pct=95.4,HIGH); Time step(pct=83.3,NOT_EXTREME); TMP4(pct=36.0,NOT_EXTREME)
  most_attributed_temporal (Notebook 07): TMP5(-0.381); TMP1(+0.000); TMP2(+0.000)
  most_unu

## 11. Limitations

1. **Class 3 labels are unknown** — the reference populations above never include Class-3 data.
2. **Percentile/unusualness describes rarity relative to labelled data, not real-world illicit
   status.** A HIGH or LOW flag is a statement about the reference distribution, not a verdict.
3. **Rarity is not evidence of illicit activity on its own** — Section 9 deliberately separates
   rarity from Class-1 association, from model sensitivity, and from forensic interpretation to
   avoid the "rare = suspicious" fallacy.
4. **Class-1/Class-2 comparisons show association in labelled data, not causation, and do not
   transfer automatically to any specific Class-3 transaction.**
5. **The overall labelled population combines Class 1 and Class 2 by simple union.** Because Class
   2 (licit) is roughly 9.2x larger than Class 1 (illicit) in this dataset, the "Overall Labelled"
   percentile is numerically dominated by licit-transaction behaviour — this is stated explicitly
   so it is not mistaken for a balanced reference.
6. **This notebook does not implement or claim any formal causal or attribution method.** Model
   sensitivity (Notebook 07) and reference unusualness (this notebook) are reported side by side,
   not combined into a single score.
7. **Final forensic interpretation is a human judgement task**, informed by, but not automated
   from, the descriptive statistics produced here.

## 12. Final verification and completion summary

In [12]:
print("\n" + "="*70); print("FINAL VERIFICATION"); print("="*70)

assert cont_stats_df[["count"]].notna().all().all()
assert not np.isinf(cont_stats_df[["mean","median","std","min","max","p25","p75","p90","p95","p99"]].to_numpy(dtype=float)).any()
print("[PASS] No infinite reference statistics.")
for txid in representative_txids + top20_txids:
    assert txid in risk_scores["txId"].values, f"{txid} not a valid Class-3 transaction in frozen scores!"
print("[PASS] All analyzed txIds are valid, frozen Class-3 transactions.")

print(f"\nREFERENCE-BASED BEHAVIOURAL UNUSUALNESS ANALYSIS COMPLETE")
print(f"Reference population: {len(labelled)} labelled (Class1={len(class1_illicit)}, Class2={len(class2_licit)})")
print(f"Representative cases analyzed in detail: {len(representative_txids)}")
print(f"Top-20 cases analyzed as batch: {len(top20_txids)}")
print(f"Fingerprint prevalence -- Overall labelled: {fp_rows[0]['percentage']:.2f}%  Top-20 Class-3: {fp_rows[-1]['percentage']:.2f}%")
print(f"Output files created:")
for p in [cont_stats_path, bin_stats_path, case_analysis_path, top20_analysis_path, fingerprint_path, combined_path]:
    print(f"  - {p}")
print(f"No training, retraining, or score regeneration occurred in this notebook.")
print(f"\nTotal runtime: {time.time()-t0:.1f}s")


FINAL VERIFICATION
[PASS] No infinite reference statistics.
[PASS] All analyzed txIds are valid, frozen Class-3 transactions.

REFERENCE-BASED BEHAVIOURAL UNUSUALNESS ANALYSIS COMPLETE
Reference population: 46564 labelled (Class1=4545, Class2=42019)
Representative cases analyzed in detail: 8
Top-20 cases analyzed as batch: 20
Fingerprint prevalence -- Overall labelled: 7.93%  Top-20 Class-3: 100.00%
Output files created:
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\reference_population_statistics.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\reference_binary_statistics.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\case_feature_percentile_analysis.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top20_batch_percentile_analysis.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\fingerprint_reference_analysis.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\co

---

# Single Transaction Forensic Walkthrough

A presentation-ready demonstration of the complete pipeline using **one real Class-3 transaction**:
`DEMO_TX_ID = 196098540`. This section only **reads** already-frozen outputs from Notebooks 05–08
— it does not retrain the model, does not refit preprocessing, does not recompute the threshold,
and does not change any existing metric or conclusion. Every displayed value is retrieved and
verified from existing files, not invented.

**Flow demonstrated:** raw transaction → transaction-time-safe behavioural features → frozen
GraphSAGE model → illicit-risk score → threshold decision → behavioural comparison with labelled
reference data → model attribution → final forensic interpretation.

## 1. Transaction identification

Verifies the transaction exists in the raw dataset as Class 3, then retrieves its stored score,
Time step, and risk category from the frozen `unknown_transaction_risk_scores.csv`. Because the
displayed score can saturate near 1.0 (as established in Notebook 07), the underlying logits are
also shown to demonstrate the output is not an exact certainty.

In [13]:
import os, json
import numpy as np
import pandas as pd

OUTPUTS_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs"
ARTIFACT_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42"
DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"

DEMO_TX_ID = 196098540

validation = {"found": False, "score": False, "attribution": False, "reference": False}

print("="*70); print("1. TRANSACTION IDENTIFICATION"); print("="*70)

tx_classes = pd.read_csv(DATA_DIR + r"\txs_classes.csv")
class_row = tx_classes[tx_classes["txId"] == DEMO_TX_ID]
assert len(class_row) == 1, f"txId {DEMO_TX_ID} not found in raw dataset!"
raw_class = int(class_row["class"].iloc[0])
assert raw_class == 3, f"txId {DEMO_TX_ID} is not Class 3 (found class={raw_class})!"
validation["found"] = True
print(f"[VERIFIED] txId {DEMO_TX_ID} exists in the raw dataset with class={raw_class} (Class 3 = Unknown).")

risk_scores = pd.read_csv(os.path.join(OUTPUTS_DIR, "unknown_transaction_risk_scores.csv"))
score_row = risk_scores[risk_scores["txId"] == DEMO_TX_ID]
assert len(score_row) == 1, "Transaction not found in frozen Class-3 scores!"
stored_score = float(score_row["illicit_risk_score"].iloc[0])
stored_time_step = int(score_row["Time step"].iloc[0])
stored_risk_level = score_row["risk_level"].iloc[0]
validation["score"] = True

with open(os.path.join(ARTIFACT_DIR, "metadata.json")) as f:
    metadata = json.load(f)
VALIDATED_THRESHOLD = metadata["selected_threshold"]

attributions = pd.read_csv(os.path.join(OUTPUTS_DIR, "top_forensic_model_attributions.csv"))
attr_rows_this_tx = attributions[attributions["txId"] == DEMO_TX_ID]
if len(attr_rows_this_tx) > 0:
    illicit_logit = float(attr_rows_this_tx["illicit_logit"].iloc[0])
    licit_logit = float(attr_rows_this_tx["licit_logit"].iloc[0])
    logit_margin = float(attr_rows_this_tx["logit_margin"].iloc[0])
    logits_available = True
else:
    illicit_logit = licit_logit = logit_margin = None
    logits_available = False

print(f"\nTransaction ID:              {DEMO_TX_ID}")
print(f"Time step:                   {stored_time_step}")
print(f"Ground-truth label:          Unknown (dataset class 3 -- never treated as licit or illicit)")
print(f"Model-based illicit-risk score (stored, from frozen inference): {stored_score}")
print(f"Displayed at 4 decimals:     {stored_score:.4f}   <-- can saturate near 1.0 due to softmax")
if logits_available:
    print(f"Underlying raw logits:       illicit={illicit_logit:.6f}  licit={licit_logit:.6f}  margin={logit_margin:.6f}")
    print("(These logits show the model's output is not an exact certainty -- softmax compresses a")
    print(" large but finite logit margin into a probability that displays as ~1.0.)")
print(f"Validated decision threshold: {VALIDATED_THRESHOLD}")
print(f"Final risk category:         {stored_risk_level} (score {'>=' if stored_score>=VALIDATED_THRESHOLD else '<'} threshold)")

1. TRANSACTION IDENTIFICATION
[VERIFIED] txId 196098540 exists in the raw dataset with class=3 (Class 3 = Unknown).

Transaction ID:              196098540
Time step:                   46
Ground-truth label:          Unknown (dataset class 3 -- never treated as licit or illicit)
Model-based illicit-risk score (stored, from frozen inference): 0.99999976
Displayed at 4 decimals:     1.0000   <-- can saturate near 1.0 due to softmax
Underlying raw logits:       illicit=3.015475  licit=-12.038462  margin=15.053937
(These logits show the model's output is not an exact certainty -- softmax compresses a
 large but finite logit margin into a probability that displays as ~1.0.)
Validated decision threshold: 0.9
Final risk category:         HIGH (score >= threshold)


## 2. Raw / behavioural profile

Retrieves the actual feature values for this transaction from `top20_forensic_case_profiles.csv`
(Notebook 06). **A verification note is printed explicitly**: several values in the original
request did not match what is stored (the request's "TMP5" and "TMP7" values were transposed with
TMP2 and NET2) — the correct, stored values are used throughout the rest of this section.

In [14]:
print("\n" + "="*70); print("2. RAW / BEHAVIOURAL PROFILE"); print("="*70)

profiles = pd.read_csv(os.path.join(OUTPUTS_DIR, "top20_forensic_case_profiles.csv"))
prow = profiles[profiles["txId"] == DEMO_TX_ID]
assert len(prow) == 1, "Transaction not found in top20_forensic_case_profiles.csv!"
prow = prow.iloc[0]

FINANCIAL_COLS = ["total_BTC","fees","size","FIN3","FIN4","FIN5","FIN6"]
STRUCTURAL_COLS = ["num_input_addresses","num_output_addresses","STR1","STR2","in_txs_degree","out_txs_degree"]
TEMPORAL_COLS = ["TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance"]
NETWORK_COLS = ["NET1","NET2"]

def show_group(name, cols):
    print(f"\n{name}:")
    for c in cols:
        v = prow[c]
        disp = "N/A (no qualifying prior activity)" if pd.isna(v) else v
        print(f"  {c:28s} = {disp}")

show_group("Financial", FINANCIAL_COLS)
show_group("Structural", STRUCTURAL_COLS)
show_group("Temporal", TEMPORAL_COLS)
show_group("Network", NETWORK_COLS)

print("\n[VERIFICATION NOTE] Values claimed in the request vs. what is actually stored for this txId:")
claimed_vs_actual = [
    ("Time step", 46, prow["Time step"]),
    ("num_input_addresses", 1, prow["num_input_addresses"]),
    ("num_output_addresses", 1, prow["num_output_addresses"]),
    ("TMP6_is_first_appearance", 1, prow["TMP6_is_first_appearance"]),
    ("TMP5 (prior receiver tx, as claimed)", 38, prow["TMP5"]),
    ("TMP7 (distinct active time steps, as claimed)", 320, prow["TMP7"]),
    ("NET2", 320, prow["NET2"]),
]
for label, claimed, actual in claimed_vs_actual:
    match = "MATCH" if claimed == actual else "MISMATCH -- using verified stored value"
    print(f"  {label}: claimed={claimed}  stored={actual}  -> {match}")
print("  ACTUAL: TMP2 (output-address age) = 38, TMP5 (prior receiver tx) = 320, TMP7 (distinct active steps) = 1.")
print("  The claimed TMP5/TMP7 values were swapped with TMP2/NET2 in the request. Verified stored values are used throughout below.")


2. RAW / BEHAVIOURAL PROFILE

Financial:
  total_BTC                    = 0.00505475
  fees                         = 0.0008816
  size                         = 221.0
  FIN3                         = 1.0
  FIN4                         = 1.0
  FIN5                         = 0.17441021
  FIN6                         = 3.98914e-06

Structural:
  num_input_addresses          = 1.0
  num_output_addresses         = 1.0
  STR1                         = 1.0
  STR2                         = 0
  in_txs_degree                = 1.0
  out_txs_degree               = 0.0

Temporal:
  TMP1                         = 0
  TMP2                         = 38
  TMP4                         = 0
  TMP5                         = 320
  TMP6                         = N/A (no qualifying prior activity)
  TMP7                         = 1
  TMP6_is_first_appearance     = 1

Network:
  NET1                         = 0
  NET2                         = 320

[VERIFICATION NOTE] Values claimed in the request vs. what is

## 3. Model detection

In [15]:
print("\n" + "="*70); print("3. MODEL DETECTION"); print("="*70)
print("The frozen GraphSAGE model assigns this unknown transaction a high model-based illicit-risk score.")
print(f"  Risk score:  {stored_score}")
print(f"  Threshold:   {VALIDATED_THRESHOLD}")
print(f"  Decision:    {'HIGH RISK' if stored_score >= VALIDATED_THRESHOLD else 'NOT HIGH RISK'}")
print("\nThis score represents similarity to patterns learned from the labelled transactions.")
print("It is not proof that the transaction is illicit.")


3. MODEL DETECTION
The frozen GraphSAGE model assigns this unknown transaction a high model-based illicit-risk score.
  Risk score:  0.99999976
  Threshold:   0.9
  Decision:    HIGH RISK

This score represents similarity to patterns learned from the labelled transactions.
It is not proof that the transaction is illicit.


## 4. Reference-population comparison

Reuses the already-computed Notebook 08 percentile analysis and fingerprint statistics for this
exact transaction — nothing is recalculated with a different method.

In [16]:
print("\n" + "="*70); print("4. REFERENCE-POPULATION COMPARISON"); print("="*70)

case_pct_this = case_analysis_df[case_analysis_df["txId"] == DEMO_TX_ID]
assert len(case_pct_this) > 0, "Transaction not found in Notebook 08 percentile analysis!"
validation["reference"] = True

print("Reference populations (Class 3 never included): Overall Labelled (46,564) | Class 1 illicit (4,545) | Class 2 licit (42,019)\n")
key_feats_for_display = ["FIN3","FIN4","FIN5","FIN6","num_input_addresses","num_output_addresses","STR1",
                          "TMP2","TMP5","TMP7","Time step","NET1","NET2"]
disp = case_pct_this[case_pct_this["feature"].isin(key_feats_for_display)][
    ["feature","feature_category","value","percentile_overall_labelled","unusualness","class1_median","class2_median"]
]
print(disp.to_string(index=False))

print("\nBinary fingerprint (1 input address + 1 output address + first-ever appearance):")
print(fingerprint_df.round(4).to_string(index=False))
print("\nInterpretation: this behavioural fingerprint is strongly associated with the labelled Class 1")
print("(illicit) population in this dataset. This is an observed association in labelled data, not")
print("proof of illicit behaviour for this (unlabelled) Class-3 transaction.")


4. REFERENCE-POPULATION COMPARISON
Reference populations (Class 3 never included): Overall Labelled (46,564) | Class 1 illicit (4,545) | Class 2 licit (42,019)

             feature feature_category     value  percentile_overall_labelled unusualness  class1_median  class2_median
                FIN3        FINANCIAL       1.0                    73.284830 NOT_EXTREME       1.000000   9.998311e-01
                FIN4        FINANCIAL       1.0                    93.778912 NOT_EXTREME       1.000000   9.376747e-01
                FIN5        FINANCIAL   0.17441                    96.016940        HIGH       0.002299   7.996288e-04
                FIN6        FINANCIAL  0.000004                    94.466283 NOT_EXTREME       0.000001   8.983726e-07
 num_input_addresses       STRUCTURAL       1.0                    26.715170 NOT_EXTREME       1.000000   2.000000e+00
num_output_addresses       STRUCTURAL       1.0                     6.206971 NOT_EXTREME       1.000000   2.000000e+00
     

## 5. Model attribution

Retrieves the Notebook 07 attribution rows for this transaction, **verifies every row actually
belongs to it**, and shows the top model-sensitive named-category features with an explicit
zero-attribution caveat.

In [17]:
print("\n" + "="*70); print("5. MODEL ATTRIBUTION"); print("="*70)

attr_this = attributions[attributions["txId"] == DEMO_TX_ID].copy()
assert (attr_this["txId"] == DEMO_TX_ID).all(), "Attribution rows do not all correspond to the demo transaction!"
validation["attribution"] = True
print(f"[VERIFIED] {len(attr_this)} attribution rows retrieved, all confirmed to belong to txId {DEMO_TX_ID}.")

named_cats = ["FINANCIAL","STRUCTURAL","TEMPORAL","NETWORK"]
attr_named = attr_this[attr_this["feature_category"].isin(named_cats)].sort_values("absolute_attribution", ascending=False)

attr_table_rows = []
for _, r in attr_named.head(10).iterrows():
    if r["attribution_delta"] == 0.0 and abs(r["original_value"] - r["perturbed_value"]) < 1e-9:
        interp = "Zero measured attribution: observed value already equals the reference baseline for this feature. This does not imply the feature is globally irrelevant."
    elif r["attribution_delta"] < 0:
        interp = "Observed value pushes the illicit-class logit higher than the reference baseline would."
    else:
        interp = "Observed value pushes the illicit-class logit lower (more licit-like) than the reference baseline."
    attr_table_rows.append({
        "Feature": r["feature"], "Category": r["feature_category"],
        "Observed value (transformed)": round(r["original_value"], 4),
        "Reference value (transformed)": round(r["perturbed_value"], 4),
        "Attribution (logit change)": round(r["attribution_delta"], 4),
        "Interpretation": interp,
    })
attr_display_df = pd.DataFrame(attr_table_rows)
print(attr_display_df.to_string(index=False))
print("\nAttribution here means model sensitivity under feature perturbation. It does not establish causality.")

summ_row = attribution_summary[attribution_summary["txId"] == DEMO_TX_ID].iloc[0]
print(f"\nCategory-level top features (from forensic_case_attribution_summary.csv, Notebook 07):")
print(f"  Financial:  {summ_row['top_financial_features']}")
print(f"  Structural: {summ_row['top_structural_features']}")
print(f"  Temporal:   {summ_row['top_temporal_features']}")
print(f"  Network:    {summ_row['top_network_features']}")


5. MODEL ATTRIBUTION
[VERIFIED] 186 attribution rows retrieved, all confirmed to belong to txId 196098540.
             Feature   Category  Observed value (transformed)  Reference value (transformed)  Attribution (logit change)                                                                                     Interpretation
                TMP5   TEMPORAL                        3.1497                        -5.1993                     -0.4102            Observed value pushes the illicit-class logit higher than the reference baseline would.
                TMP2   TEMPORAL                        5.1993                        -5.1993                      0.2065 Observed value pushes the illicit-class logit lower (more licit-like) than the reference baseline.
 num_input_addresses STRUCTURAL                       -5.1993                         0.2953                     -0.1843            Observed value pushes the illicit-class logit higher than the reference baseline would.
            

## 6. Unusualness vs. model sensitivity — five distinct dimensions

In [18]:
print("\n" + "="*70); print("6. FIVE DISTINCT DIMENSIONS -- NOT TO BE CONFLATED"); print("="*70)

concept_table = pd.DataFrame([
    {"Dimension": "Model risk", "What it tells us": "How strongly the frozen model associates the transaction with the learned illicit pattern."},
    {"Dimension": "Behavioural unusualness", "What it tells us": "How extreme the transaction's behaviour is relative to the labelled reference population."},
    {"Dimension": "Class association", "What it tells us": "How prevalent that behaviour is in labelled Class 1 versus Class 2."},
    {"Dimension": "Model sensitivity", "What it tells us": "Whether perturbing the feature changes the frozen model output."},
    {"Dimension": "Forensic interpretation", "What it tells us": "What these pieces of evidence collectively suggest for investigation/prioritization."},
])
print(concept_table.to_string(index=False))
print("\nThese are DIFFERENT concepts. None of them alone establishes illicit activity.")


6. FIVE DISTINCT DIMENSIONS -- NOT TO BE CONFLATED
              Dimension                                                                           What it tells us
             Model risk How strongly the frozen model associates the transaction with the learned illicit pattern.
Behavioural unusualness  How extreme the transaction's behaviour is relative to the labelled reference population.
      Class association                        How prevalent that behaviour is in labelled Class 1 versus Class 2.
      Model sensitivity                            Whether perturbing the feature changes the frozen model output.
Forensic interpretation       What these pieces of evidence collectively suggest for investigation/prioritization.

These are DIFFERENT concepts. None of them alone establishes illicit activity.


## 7. Final forensic summary

In [19]:
print("\n" + "="*70); print("7. FINAL FORENSIC SUMMARY"); print("="*70)

tmp2_val = int(prow["TMP2"])
tmp5_val = int(prow["TMP5"])
net2_val = int(prow["NET2"])
fp_top20_pct = fingerprint_df.loc[fingerprint_df["population"]=="Top20_Class3","percentage"].iloc[0]
fp_overall_pct = fingerprint_df.loc[fingerprint_df["population"]=="Overall_Labelled","percentage"].iloc[0]
fp_c1_pct = fingerprint_df.loc[fingerprint_df["population"]=="Class1_Illicit","percentage"].iloc[0]
fp_c2_pct = fingerprint_df.loc[fingerprint_df["population"]=="Class2_Licit","percentage"].iloc[0]

summary_text = (
    f"Transaction {DEMO_TX_ID} received a high model-based illicit-risk score of {stored_score:.6f}, "
    f"exceeding the validated threshold of {VALIDATED_THRESHOLD}.\n\n"
    f"Its observed behavioural profile includes a single input address, a single output address, and "
    f"first-ever appearance in the dataset at time step {stored_time_step}, with an output-address age "
    f"(TMP2) of {tmp2_val}, {tmp5_val} prior receiver transactions (TMP5), and {net2_val} historical "
    f"output-linked transactions (NET2).\n\n"
    f"Relative to the labelled reference population, this 1-input/1-output/first-appearance combination "
    f"occurs in {fp_overall_pct:.2f}% of overall labelled transactions, but in {fp_c1_pct:.2f}% of labelled "
    f"illicit transactions versus only {fp_c2_pct:.2f}% of labelled licit transactions -- a strong "
    f"association with the labelled illicit class, though not proof for this specific unlabelled case. "
    f"TMP2 and TMP5 are both at or above the 99.9th percentile of the overall labelled population.\n\n"
    f"Model perturbation analysis indicates sensitivity to {summ_row['top_financial_features'].split(';')[0].strip()}, "
    f"{summ_row['top_structural_features'].split(';')[0].strip()}, and {summ_row['top_temporal_features'].split(';')[0].strip()}.\n\n"
    f"Taken together, these observations make the transaction suitable for forensic prioritization and "
    f"further investigation. They do not establish that the transaction is illicit, because Class 3 has "
    f"no ground-truth label in this analysis."
)
print(summary_text)


7. FINAL FORENSIC SUMMARY
Transaction 196098540 received a high model-based illicit-risk score of 1.000000, exceeding the validated threshold of 0.9.

Its observed behavioural profile includes a single input address, a single output address, and first-ever appearance in the dataset at time step 46, with an output-address age (TMP2) of 38, 320 prior receiver transactions (TMP5), and 320 historical output-linked transactions (NET2).

Relative to the labelled reference population, this 1-input/1-output/first-appearance combination occurs in 7.93% of overall labelled transactions, but in 63.74% of labelled illicit transactions versus only 1.90% of labelled licit transactions -- a strong association with the labelled illicit class, though not proof for this specific unlabelled case. TMP2 and TMP5 are both at or above the 99.9th percentile of the overall labelled population.

Model perturbation analysis indicates sensitivity to FIN6(-0.182), num_input_addresses(-0.184), and TMP5(-0.410).

T

## 8. Presentation-ready visual panel

In [20]:
print("\n" + "="*70); print("8. PRESENTATION PANEL"); print("="*70)

panel = f"""
==================================================
        FORENSIC CASE: {DEMO_TX_ID}
==================================================

Model Risk Score:        {stored_score:.6f}
Threshold:                0.90
Risk Category:            {stored_risk_level}
Dataset Label:             UNKNOWN (Class 3)
Time Step:                 {stored_time_step}

Behavioural fingerprint:
  - 1 input address
  - 1 output address
  - First-ever appearance
  - Output-address age (TMP2): {tmp2_val}
  - Prior receiver transactions (TMP5): {tmp5_val}
  - Historical output-linked transactions (NET2): {net2_val}

Most relevant model-sensitive features:
  - Financial:  {summ_row['top_financial_features']}
  - Structural: {summ_row['top_structural_features']}
  - Temporal:   {summ_row['top_temporal_features']}
  - Network:    {summ_row['top_network_features']}

Reference-population evidence:
  - 1-in/1-out/first-appearance fingerprint: {fp_top20_pct:.0f}% of top-20 Class-3 candidates
    vs. {fp_overall_pct:.2f}% overall labelled ({fp_c1_pct:.2f}% Class 1 illicit, {fp_c2_pct:.2f}% Class 2 licit)
  - TMP2, TMP5, FIN5, Time step: HIGH unusualness (>=95th percentile of labelled population)

Forensic interpretation:
  Suitable for prioritization / further investigation
  NOT proof of illicit activity
==================================================
"""
print(panel)


8. PRESENTATION PANEL

        FORENSIC CASE: 196098540

Model Risk Score:        1.000000
Threshold:                0.90
Risk Category:            HIGH
Dataset Label:             UNKNOWN (Class 3)
Time Step:                 46

Behavioural fingerprint:
  - 1 input address
  - 1 output address
  - First-ever appearance
  - Output-address age (TMP2): 38
  - Prior receiver transactions (TMP5): 320
  - Historical output-linked transactions (NET2): 320

Most relevant model-sensitive features:
  - Financial:  FIN6(-0.182); FIN3(-0.091); size(-0.080)
  - Structural: num_input_addresses(-0.184); out_txs_degree(-0.073); num_output_addresses(-0.047)
  - Temporal:   TMP5(-0.410); TMP2(+0.207); TMP1(+0.000)
  - Network:    NET2(-0.075); NET1(+0.000)

Reference-population evidence:
  - 1-in/1-out/first-appearance fingerprint: 100% of top-20 Class-3 candidates
    vs. 7.93% overall labelled (63.74% Class 1 illicit, 1.90% Class 2 licit)
  - TMP2, TMP5, FIN5, Time step: HIGH unusualness (>=95th perc

## 9. "How the system detected it" — end-to-end flow

In [21]:
print("\n" + "="*70); print("9. HOW THE SYSTEM DETECTED IT"); print("="*70)
flow = """
Transaction
  |
  v
Transaction-time-safe feature extraction (14 engineered features + A2 inherited features)
  |
  v
Feature preprocessing (train-only median imputation + QuantileTransformer, saved artifacts only)
  |
  v
GraphSAGE + transaction network (203,769 nodes, 234,355 directed edges, 2-layer message passing)
  |
  v
Illicit-risk score (softmax output)
  |
  v
Validated threshold (0.90, fixed from labelled-data protocol)
  |
  v
High-risk unknown transaction (Class 3, screened -- not confirmed)
  |
  v
Behavioural reference comparison (Notebook 08 -- labelled Class 1 / Class 2 percentiles)
  |
  v
Model attribution (Notebook 07 -- feature-occlusion sensitivity)
  |
  v
Forensic prioritization (human review, not automated verdict)
"""
print(flow)


9. HOW THE SYSTEM DETECTED IT

Transaction
  |
  v
Transaction-time-safe feature extraction (14 engineered features + A2 inherited features)
  |
  v
Feature preprocessing (train-only median imputation + QuantileTransformer, saved artifacts only)
  |
  v
GraphSAGE + transaction network (203,769 nodes, 234,355 directed edges, 2-layer message passing)
  |
  v
Illicit-risk score (softmax output)
  |
  v
Validated threshold (0.90, fixed from labelled-data protocol)
  |
  v
High-risk unknown transaction (Class 3, screened -- not confirmed)
  |
  v
Behavioural reference comparison (Notebook 08 -- labelled Class 1 / Class 2 percentiles)
  |
  v
Model attribution (Notebook 07 -- feature-occlusion sensitivity)
  |
  v
Forensic prioritization (human review, not automated verdict)



## 10. Validation summary

In [22]:
print("\n" + "="*70); print("10. VALIDATION SUMMARY"); print("="*70)
print(f"Demo transaction found:      {'YES' if validation['found'] else 'NO'}")
print(f"Model score retrieved:       {'YES' if validation['score'] else 'NO'}")
print(f"Attribution retrieved:       {'YES' if validation['attribution'] else 'NO'}")
print(f"Reference analysis retrieved:{'YES' if validation['reference'] else 'NO'}")
print(f"Threshold unchanged:         YES (0.90, read from frozen metadata.json)")
print(f"Model retrained:             NO")
print(f"Existing artifacts modified: NO")


10. VALIDATION SUMMARY
Demo transaction found:      YES
Model score retrieved:       YES
Attribution retrieved:       YES
Reference analysis retrieved:YES
Threshold unchanged:         YES (0.90, read from frozen metadata.json)
Model retrained:             NO
Existing artifacts modified: NO


---

# Interactive Single Transaction Forensic Analyzer

A **reusable** forensic analyzer for any transaction ID (Class 1, Class 2, or Class 3), built on
top of everything already established in this notebook and Notebooks 05–07. Change only
`DEMO_TX_ID` below and rerun — nothing else needs to change.

**What is reused, unchanged:**
- The frozen GraphSAGE checkpoint and preprocessing artifacts (loaded read-only, used only for a
  forward pass — no training, no refitting).
- The already-validated 0.90 threshold.
- The existing reference-population definitions and statistics from this notebook (`labelled`,
  `class1_illicit`, `class2_licit`, `cont_stats_df`, `bin_stats_df`, `build_case_rows`,
  `fingerprint_df`).
- The existing attribution results (`top_forensic_model_attributions.csv`) — looked up, never
  recomputed for a new transaction.

**What "live" means here:** the model score for an *arbitrary* transaction (including Class 1/2,
which were never scored in `unknown_transaction_risk_scores.csv`) is obtained by running the
already-frozen model in `eval()` mode over the full graph once — this is inference, not training,
and is verified below to exactly reproduce the frozen Class-3 scores already on file.

## Load frozen model for live inference (read-only, no training)

Loads the same checkpoint, preprocessing artifacts, and graph construction used in Notebooks 05
and 07. Runs **one** forward pass over the full graph and cross-checks the result against the
frozen `unknown_transaction_risk_scores.csv` to confirm it reproduces those values exactly before
using it for arbitrary transactions.

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import joblib

print("="*70); print("LOAD FROZEN MODEL FOR LIVE INFERENCE (no training)"); print("="*70)

checkpoint = torch.load(os.path.join(ARTIFACT_DIR, "model_checkpoint.pt"), weights_only=False)
arch = checkpoint["architecture"]

class GraphSAGENet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

model = GraphSAGENet(in_dim=arch["in_dim"], hidden_dim=arch["hidden_dim"], out_dim=arch["out_dim"], dropout=arch["dropout"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
VALIDATED_THRESHOLD = checkpoint["selected_threshold"]

qt = joblib.load(os.path.join(ARTIFACT_DIR, "quantile_transformer.joblib"))
median_artifact = joblib.load(os.path.join(ARTIFACT_DIR, "median_imputation.joblib"))
augmented_medians_qt = pd.Series(median_artifact["augmented_medians"])
new_feature_medians_qt = pd.Series(median_artifact["new_feature_medians"])
continuous_new_qt = median_artifact["continuous_new_features"]
scaling_cols_b = a2_cols + continuous_new_qt
with open(os.path.join(ARTIFACT_DIR, "scaling_columns.json")) as f:
    saved_scaling_cols = json.load(f)
assert scaling_cols_b == saved_scaling_cols

edges_raw = pd.read_csv(DATA_DIR + r"\txs_edgelist.csv")
txid_to_node = {txid: i for i, txid in enumerate(tx_data["txId"].values)}
edge_cols = edges_raw.columns.tolist()
src = edges_raw[edge_cols[0]].map(txid_to_node).values
dst = edges_raw[edge_cols[1]].map(txid_to_node).values
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
assert edge_index.shape[1] == 234355

X_b = merged.copy()
for col in augmented_features: X_b[col] = X_b[col].fillna(augmented_medians_qt[col])
for col in continuous_new_qt: X_b[col] = X_b[col].fillna(new_feature_medians_qt[col])
X_b_q = X_b.copy()
X_b_q[scaling_cols_b] = X_b_q[scaling_cols_b].astype("float64")
X_b_q[scaling_cols_b] = qt.transform(X_b[scaling_cols_b])
x_infer = torch.tensor(X_b_q[final_feature_cols_b].astype("float32").values, dtype=torch.float32)
assert x_infer.shape == (203769, 186)

with torch.no_grad():
    full_logits = model(x_infer, edge_index)
assert torch.isfinite(full_logits).all().item()
full_probs = F.softmax(full_logits, dim=1)[:, 1].numpy()
full_illicit_logit = full_logits[:, 1].numpy()
full_licit_logit = full_logits[:, 0].numpy()

check_ids = risk_scores["txId"].values[:200]
check_idx = [txid_to_node[t] for t in check_ids]
recomputed = full_probs[check_idx]
stored = risk_scores.set_index("txId").loc[check_ids, "illicit_risk_score"].values
max_diff = np.abs(recomputed - stored).max()
print(f"Sanity check: max |recomputed - frozen| score diff over 200 Class-3 txIds = {max_diff:.2e} (must be ~0)")
assert max_diff < 1e-5
print("[PASS] Live inference (frozen model, no training) exactly reproduces the frozen Class-3 scores.")
print(f"Model + graph loaded for live inference: {x_infer.shape[0]} nodes, {edge_index.shape[1]} edges, threshold={VALIDATED_THRESHOLD}")

LOAD FROZEN MODEL FOR LIVE INFERENCE (no training)
Sanity check: max |recomputed - frozen| score diff over 200 Class-3 txIds = 2.94e-08 (must be ~0)
[PASS] Live inference (frozen model, no training) exactly reproduces the frozen Class-3 scores.
Model + graph loaded for live inference: 203769 nodes, 234355 edges, threshold=0.9


In [24]:
pip install torch_geometric

## Feature dictionary

Human-readable names, formulas (where applicable), meanings, and forensic interpretations for
every one of the 14 engineered features, using only meanings already established in this project
(`03_Transaction_Time_Safe_Features.ipynb`). `FIN1` was considered during design but deferred/not
implemented; `FIN2` is not part of the final 186-feature matrix — neither is shown as implemented.

In [25]:
FEATURE_DICTIONARY = {
    "FIN3": {"name": "Largest Input Share", "formula": "in_BTC_max / in_BTC_total",
             "meaning": "The fraction of the transaction's total input BTC contributed by the largest input.",
             "forensic": "Indicates whether funding is concentrated in one major input or distributed across multiple inputs."},
    "FIN4": {"name": "Largest Output Share", "formula": "out_BTC_max / out_BTC_total",
             "meaning": "The fraction of the transaction's total output BTC represented by the largest output.",
             "forensic": "Indicates whether value is concentrated in one output or distributed across multiple outputs."},
    "FIN5": {"name": "Fee-to-Value Ratio", "formula": "fees / total_BTC",
             "meaning": "Transaction fee relative to the transaction's BTC value.",
             "forensic": "Describes the relative transaction cost compared with the transferred value."},
    "FIN6": {"name": "Fee Rate", "formula": "fees / size",
             "meaning": "Transaction fee relative to transaction size.",
             "forensic": "Describes the transaction's fee intensity."},
    "STR1": {"name": "Input/Output Address Ratio", "formula": "num_input_addresses / num_output_addresses",
             "meaning": "Number of input addresses relative to number of output addresses.",
             "forensic": "Describes the transaction's input/output structure or fan-in/fan-out shape."},
    "STR2": {"name": "Same-Transaction Address Reuse", "formula": None,
             "meaning": "Whether an address occurs on both the input and output side of the same transaction.",
             "forensic": "Identifies same-transaction address reuse."},
    "TMP1": {"name": "Input Address Age", "formula": None,
             "meaning": "Age of the relevant input address activity at the transaction's time boundary (see 03_Transaction_Time_Safe_Features.ipynb for exact implementation semantics).",
             "forensic": "Reflects how long the input address has been active prior to this transaction."},
    "TMP2": {"name": "Output Address Age", "formula": None,
             "meaning": "Age of the relevant output address activity at the transaction's time boundary (see 03_Transaction_Time_Safe_Features.ipynb for exact implementation semantics).",
             "forensic": "Reflects how long the output address has been active prior to this transaction."},
    "TMP4": {"name": "Prior Sender Transactions", "formula": None,
             "meaning": "Historical transaction count associated with the input-address side up to the transaction's time boundary.",
             "forensic": "Reflects the input address's prior transactional activity."},
    "TMP5": {"name": "Prior Receiver Transactions", "formula": None,
             "meaning": "Historical transaction count associated with the output-address side up to the transaction's time boundary.",
             "forensic": "Reflects the output address's prior transactional activity."},
    "TMP6": {"name": "Time Since Last Activity", "formula": None,
             "meaning": "Time elapsed since the relevant address was previously active.",
             "forensic": "Reflects recency of prior activity for the relevant address."},
    "TMP7": {"name": "Distinct Active Time Steps", "formula": None,
             "meaning": "Number of distinct time steps in which the relevant address has appeared up to the transaction's time boundary.",
             "forensic": "Reflects the breadth of the address's activity history."},
    "NET1": {"name": "Historical Input-Side Linked Transactions", "formula": None,
             "meaning": "Number of other historical transactions connected through shared input addresses (existing NET1 implementation).",
             "forensic": "Reflects the address's historical transactional network on the input side."},
    "NET2": {"name": "Historical Output-Side Linked Transactions", "formula": None,
             "meaning": "Number of other historical transactions connected through shared output addresses (existing NET2 implementation).",
             "forensic": "Reflects the address's historical transactional network on the output side."},
}
NOT_IMPLEMENTED_NOTE = ("FIN1 was considered during feature design but was deferred/rejected from the final "
                         "implemented feature set and is NOT part of the 186-feature model. FIN2 is not present "
                         "in the final feature matrix. Neither is displayed as an implemented feature below.")

OTHER_DISPLAY_NAMES = {
    "total_BTC": "Total BTC Value", "fees": "Transaction Fees (BTC)", "size": "Transaction Size (bytes)",
    "num_input_addresses": "Input Address Count", "num_output_addresses": "Output Address Count",
    "in_txs_degree": "Incoming Transaction Degree", "out_txs_degree": "Outgoing Transaction Degree",
    "TMP6_is_first_appearance": "First-Ever Appearance", "Time step": "Time Step",
}
def display_name(code):
    if code in FEATURE_DICTIONARY: return f"{FEATURE_DICTIONARY[code]['name']} ({code})"
    if code in OTHER_DISPLAY_NAMES: return OTHER_DISPLAY_NAMES[code]
    return code

print("Feature dictionary loaded:", len(FEATURE_DICTIONARY), "engineered features documented.")
print(NOT_IMPLEMENTED_NOTE)
print()
for code_, d in FEATURE_DICTIONARY.items():
    print(f"{code_} -> {d['name']}" + (f"  [{d['formula']}]" if d['formula'] else ""))

Feature dictionary loaded: 14 engineered features documented.
FIN1 was considered during feature design but was deferred/rejected from the final implemented feature set and is NOT part of the 186-feature model. FIN2 is not present in the final feature matrix. Neither is displayed as an implemented feature below.

FIN3 -> Largest Input Share  [in_BTC_max / in_BTC_total]
FIN4 -> Largest Output Share  [out_BTC_max / out_BTC_total]
FIN5 -> Fee-to-Value Ratio  [fees / total_BTC]
FIN6 -> Fee Rate  [fees / size]
STR1 -> Input/Output Address Ratio  [num_input_addresses / num_output_addresses]
STR2 -> Same-Transaction Address Reuse
TMP1 -> Input Address Age
TMP2 -> Output Address Age
TMP4 -> Prior Sender Transactions
TMP5 -> Prior Receiver Transactions
TMP6 -> Time Since Last Activity
TMP7 -> Distinct Active Time Steps
NET1 -> Historical Input-Side Linked Transactions
NET2 -> Historical Output-Side Linked Transactions


## Dynamic interpretation helpers

Each helper explains **what a value means**, adapting to the actual number — never a fixed
"suspicious/not suspicious" label based only on a threshold.

In [26]:
def fmt_num(v, nd=4):
    if pd.isna(v): return None
    return round(float(v), nd)

def interp_share(v, side):
    if pd.isna(v): return "Value unavailable."
    if v >= 0.999: return f"All recorded {side} value is concentrated in one {side}."
    return f"The largest {side} contributes approximately {v*100:.1f}% of the recorded {side} value."

def interp_ratio(fee_like, denom_label):
    if pd.isna(fee_like): return "Value unavailable."
    return f"The fee represents approximately {fee_like*100:.4f}% of the transaction's {denom_label}."

def interp_count(v, singular, plural, plural_verb):
    if pd.isna(v): return "Value unavailable."
    v = int(v)
    if v == 1: return f"The transaction uses a single {singular}."
    return f"The transaction {plural_verb} {v} {plural}."

def interp_str1(v):
    if pd.isna(v): return "Value unavailable."
    if abs(v - 1.0) < 1e-9: return "Equal numbers of input and output addresses."
    if v > 1: return f"More input addresses than output addresses (fan-in shape), ratio={v:.3g}."
    return f"More output addresses than input addresses (fan-out shape), ratio={v:.3g}."

def interp_age(v, side):
    if pd.isna(v): return "Value unavailable."
    v_int = int(v) if float(v).is_integer() else v
    if v == 0: return f"The relevant {side} address shows no prior recorded activity interval before this transaction."
    return f"The relevant {side} address has approximately {v_int} time step(s) of recorded activity history before this transaction."

def interp_prior_tx(v, side):
    if pd.isna(v): return "Value unavailable."
    v = int(v)
    if v == 0: return f"No prior transactions are recorded on the {side} side up to this transaction's time boundary."
    return f"{v} prior transaction(s) are recorded on the {side} side up to this transaction's time boundary."

def interp_tmp6(v):
    if pd.isna(v):
        return "No previous activity interval is available because this is the first observed activity."
    return f"Approximately {v:.4g} time step(s) elapsed since the relevant address was previously active."

def interp_tmp7(v):
    if pd.isna(v): return "Value unavailable."
    v = int(v)
    return f"The relevant address has appeared in {v} distinct time step(s) up to this transaction's time boundary."

def interp_first_appearance(v):
    if pd.isna(v): return "Value unavailable."
    return "First observed activity in the available transaction history." if v == 1 else "This address has prior recorded activity before this transaction."

def interp_net(v, side):
    if pd.isna(v): return "Value unavailable."
    v = int(v)
    if v == 0: return f"No historical transactions are linked through shared {side}-side addresses."
    return f"Connected to {v} other historical transaction(s) through shared {side}-side addresses."

print("Dynamic interpretation helpers defined.")

Dynamic interpretation helpers defined.


## The reusable analyzer function

`analyze_transaction(tx_id)` implements the full walkthrough for any transaction ID: existence
check → overview → behavioural characteristics (human-readable) → model detection (live,
frozen-model inference) → reference-population comparison (existing methodology, applied live) →
attribution lookup (existing results only, never recomputed) → forensic interpretation → a
presentation-ready summary panel. Missing data is reported explicitly ("N/A", "unavailable", "not
found") — never silently replaced with zero or an invented value.

In [27]:

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import joblib

ARTIFACT_DIR = "C:/Users/koush/OneDrive/Documents/Financial_Forensics/outputs/gnn_B_quantile_seed42"
print("="*70); print("LOAD FROZEN MODEL FOR LIVE INFERENCE (no training)"); print("="*70)

checkpoint = torch.load(os.path.join("C:/Users/koush/OneDrive/Documents/Financial_Forensics/outputs/gnn_B_quantile_seed42", "model_checkpoint.pt"), weights_only=False)
arch = checkpoint["architecture"]

class GraphSAGENet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

model = GraphSAGENet(in_dim=arch["in_dim"], hidden_dim=arch["hidden_dim"], out_dim=arch["out_dim"], dropout=arch["dropout"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
VALIDATED_THRESHOLD = checkpoint["selected_threshold"]

qt = joblib.load(os.path.join("C:/Users/koush/OneDrive/Documents/Financial_Forensics/outputs/gnn_B_quantile_seed42", "quantile_transformer.joblib"))
median_artifact = joblib.load(os.path.join(ARTIFACT_DIR, "median_imputation.joblib"))
augmented_medians_qt = pd.Series(median_artifact["augmented_medians"])
new_feature_medians_qt = pd.Series(median_artifact["new_feature_medians"])
continuous_new_qt = median_artifact["continuous_new_features"]
scaling_cols_b = a2_cols + continuous_new_qt
with open(os.path.join(ARTIFACT_DIR, "scaling_columns.json")) as f:
    saved_scaling_cols = json.load(f)
assert scaling_cols_b == saved_scaling_cols
binary_cols_b = ["augmented_features_missing", "STR2", "TMP6_is_first_appearance"]
with open(os.path.join(ARTIFACT_DIR, "binary_columns.json")) as f:
    saved_binary_cols = json.load(f)
assert binary_cols_b == saved_binary_cols

edges_raw = pd.read_csv(DATA_DIR + r"\txs_edgelist.csv")
txid_to_node = {txid: i for i, txid in enumerate(tx_data["txId"].values)}
edge_cols = edges_raw.columns.tolist()
src = edges_raw[edge_cols[0]].map(txid_to_node).values
dst = edges_raw[edge_cols[1]].map(txid_to_node).values
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
assert edge_index.shape[1] == 234355

X_b = merged.copy()
for col in augmented_features: X_b[col] = X_b[col].fillna(augmented_medians_qt[col])
for col in continuous_new_qt: X_b[col] = X_b[col].fillna(new_feature_medians_qt[col])
X_b_q = X_b.copy()
X_b_q[scaling_cols_b] = X_b_q[scaling_cols_b].astype("float64")
X_b_q[scaling_cols_b] = qt.transform(X_b[scaling_cols_b])
x_infer = torch.tensor(X_b_q[final_feature_cols_b].astype("float32").values, dtype=torch.float32)
assert x_infer.shape == (203769, 186)

with torch.no_grad():
    full_logits = model(x_infer, edge_index)
assert torch.isfinite(full_logits).all().item()
full_probs = F.softmax(full_logits, dim=1)[:, 1].numpy()
full_illicit_logit = full_logits[:, 1].numpy()
full_licit_logit = full_logits[:, 0].numpy()

check_ids = risk_scores["txId"].values[:200]
check_idx = [txid_to_node[t] for t in check_ids]
recomputed = full_probs[check_idx]
stored = risk_scores.set_index("txId").loc[check_ids, "illicit_risk_score"].values
max_diff = np.abs(recomputed - stored).max()
print(f"Sanity check: max |recomputed - frozen| score diff over 200 Class-3 txIds = {max_diff:.2e} (must be ~0)")
assert max_diff < 1e-5
print("[PASS] Live inference (frozen model, no training) exactly reproduces the frozen Class-3 scores.")
print(f"Model + graph loaded for live inference: {x_infer.shape[0]} nodes, {edge_index.shape[1]} edges, threshold={VALIDATED_THRESHOLD}")

# ------------------------------------------------------------------
# LIVE, ON-DEMAND FEATURE-OCCLUSION ATTRIBUTION (identical method to
# Notebook 07 -- train-only baseline, single-feature perturbation,
# full-graph forward pass -- just computed on demand instead of only
# for a fixed precomputed set of transactions).
# ------------------------------------------------------------------
print("\n" + "="*70); print("PREPARE LIVE ATTRIBUTION (train-only baseline, same method as Notebook 07)"); print("="*70)

time_step_arr = tx_data["Time step"].values
def _split_of_step(t):
    if t <= 34: return 0
    elif t <= 41: return 1
    else: return 2
_split_id = np.array([_split_of_step(t) for t in time_step_arr])
_known_mask_all = tx_data["class"].isin([1, 2]).values
train_mask_np = (_split_id == 0) & _known_mask_all
assert train_mask_np.sum() == 29894
print(f"train_mask_np: {train_mask_np.sum()} rows (expected 29894)")

col_index_map = {c: i for i, c in enumerate(final_feature_cols_b)}
baseline_vector = np.zeros(186, dtype=np.float32)
_x_infer_np_train = x_infer.numpy()[train_mask_np]
for c in scaling_cols_b:
    idx = col_index_map[c]
    baseline_vector[idx] = np.median(_x_infer_np_train[:, idx])
for c in binary_cols_b:
    idx = col_index_map[c]
    vals, counts = np.unique(_x_infer_np_train[:, idx], return_counts=True)
    baseline_vector[idx] = vals[np.argmax(counts)]
print("baseline_vector computed (train-only median for scaled columns, train-only mode for binary columns).")

ATTR_FINANCIAL = ["FIN3","FIN4","FIN5","FIN6","total_BTC","fees","size","in_BTC_min","in_BTC_max","in_BTC_mean",
                   "in_BTC_median","in_BTC_total","out_BTC_min","out_BTC_max","out_BTC_mean","out_BTC_median","out_BTC_total"]
ATTR_STRUCTURAL = ["STR1","STR2","num_input_addresses","num_output_addresses","in_txs_degree","out_txs_degree"]
ATTR_TEMPORAL = ["TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance"]
ATTR_NETWORK = ["NET1","NET2"]
ATTR_ALL = ATTR_FINANCIAL + ATTR_STRUCTURAL + ATTR_TEMPORAL + ATTR_NETWORK
ATTR_CATEGORY_OF = {}
for f in ATTR_FINANCIAL: ATTR_CATEGORY_OF[f] = "FINANCIAL"
for f in ATTR_STRUCTURAL: ATTR_CATEGORY_OF[f] = "STRUCTURAL"
for f in ATTR_TEMPORAL: ATTR_CATEGORY_OF[f] = "TEMPORAL"
for f in ATTR_NETWORK: ATTR_CATEGORY_OF[f] = "NETWORK"
assert len(ATTR_ALL) == 32
print(f"Live attribution will occlude {len(ATTR_ALL)} named-category features per transaction "
      f"(FINANCIAL={len(ATTR_FINANCIAL)}, STRUCTURAL={len(ATTR_STRUCTURAL)}, TEMPORAL={len(ATTR_TEMPORAL)}, "
      f"NETWORK={len(ATTR_NETWORK)}) -- identical scope to Notebook 07's named-category attribution.")

def compute_live_attribution(node_idx):
    # Feature-occlusion attribution computed on demand for ANY node (Class 1, 2, or 3), using
    # the exact Notebook 07 methodology: train-only baseline, single-feature perturbation, full-graph
    # forward pass, illicit-logit delta. Not precomputed or cached -- runs fresh for whatever
    # transaction is requested. Takes ~7-8 seconds (32 forward passes over the full graph).
    baseline_illicit_logit_orig = float(full_illicit_logit[node_idx])
    original_row = x_infer[node_idx].clone()
    rows = []
    try:
        for feat in ATTR_ALL:
            f_idx = col_index_map[feat]
            orig_val = float(original_row[f_idx])
            pert_val = float(baseline_vector[f_idx])
            if abs(orig_val - pert_val) < 1e-9:
                delta = 0.0
            else:
                x_infer[node_idx, f_idx] = pert_val
                with torch.no_grad():
                    pert_logit = float(model(x_infer, edge_index)[node_idx, 1])
                x_infer[node_idx, f_idx] = orig_val
                delta = pert_logit - baseline_illicit_logit_orig
            rows.append({"feature": feat, "feature_category": ATTR_CATEGORY_OF[feat],
                          "original_value": orig_val, "perturbed_value": pert_val,
                          "attribution_delta": delta, "absolute_attribution": abs(delta)})
    finally:
        x_infer[node_idx] = original_row  # guarantee restoration even if an error occurs mid-loop
    assert torch.allclose(x_infer[node_idx], original_row), "Feature row not fully restored!"
    return pd.DataFrame(rows)

# Cross-validate against the stored Notebook 07 attribution for the one transaction that has both,
# to confirm the live computation is methodologically identical, not a new/different method.
_stored_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "top_forensic_model_attributions.csv"))
_stored_this = _stored_df[_stored_df["txId"] == 196098540].set_index("feature")
_live_check = compute_live_attribution(txid_to_node[196098540]).set_index("feature")
_max_diff = max(abs(_live_check.loc[f, "attribution_delta"] - _stored_this.loc[f, "attribution_delta"]) for f in ATTR_ALL)
print(f"\nCross-check: live attribution vs. stored Notebook-07 attribution for txId 196098540 -- "
      f"max diff = {_max_diff:.6f} (must be ~0)")
assert _max_diff < 1e-3
print("[PASS] Live on-demand attribution exactly reproduces the stored Notebook-07 result -- same method, just not cached.")

LOAD FROZEN MODEL FOR LIVE INFERENCE (no training)
Sanity check: max |recomputed - frozen| score diff over 200 Class-3 txIds = 2.94e-08 (must be ~0)
[PASS] Live inference (frozen model, no training) exactly reproduces the frozen Class-3 scores.
Model + graph loaded for live inference: 203769 nodes, 234355 edges, threshold=0.9

PREPARE LIVE ATTRIBUTION (train-only baseline, same method as Notebook 07)
train_mask_np: 29894 rows (expected 29894)
baseline_vector computed (train-only median for scaled columns, train-only mode for binary columns).
Live attribution will occlude 32 named-category features per transaction (FINANCIAL=17, STRUCTURAL=6, TEMPORAL=7, NETWORK=2) -- identical scope to Notebook 07's named-category attribution.

Cross-check: live attribution vs. stored Notebook-07 attribution for txId 196098540 -- max diff = 0.000000 (must be ~0)
[PASS] Live on-demand attribution exactly reproduces the stored Notebook-07 result -- same method, just not cached.


In [28]:
def analyze_transaction(tx_id):
    print("="*60)
    print("        BITCOIN TRANSACTION FORENSIC ANALYZER")
    print("="*60)
    print("[Analyzer v7 -- full per-feature forensic comparison: observed value, reference percentile,")
    print(" class medians, unusualness, and case-specific model sensitivity, shown together per feature]")
    print(f"\nTransaction ID: {tx_id}\n")

    if tx_id not in txid_to_node:
        print("Transaction ID not found in the Elliptic++ transaction dataset.")
        return {"found": False}

    node_idx = txid_to_node[tx_id]
    tx_row = tx_data.iloc[node_idx]
    tx_class = int(tx_row["class"])
    time_step = int(tx_row["Time step"])
    class_label_map = {1: "Class 1 -- Labelled Illicit", 2: "Class 2 -- Labelled Licit", 3: "Class 3 -- Unknown"}
    class_label = class_label_map.get(tx_class, f"Unrecognized class ({tx_class})")

    score = float(full_probs[node_idx])
    score_available = np.isfinite(score)

    def risk_band(s):
        if not np.isfinite(s): return "UNAVAILABLE"
        if s >= VALIDATED_THRESHOLD: return "HIGH"
        if s >= 0.50: return "MODERATE"
        return "LOW"
    band = risk_band(score)
    assessment = "HIGH RISK" if score_available and score >= VALIDATED_THRESHOLD else "BELOW THRESHOLD"

    result = {"found": True, "txId": tx_id, "class": tx_class, "time_step": time_step,
              "score": score, "risk_band": band, "assessment": assessment, "attribution_available": False,
              "reference_available": False, "n_key_contributors": 0}

    # ------------------------------------------------------------
    # TRANSACTION OVERVIEW
    # ------------------------------------------------------------
    print("="*60); print("TRANSACTION OVERVIEW"); print("="*60)
    print(f"Transaction ID:              {tx_id}")
    print(f"Dataset class:               {class_label}")
    print(f"Time step:                   {time_step}")
    print(f"Model illicit-risk score:    {score:.6f}" if score_available else "Model illicit-risk score:    UNAVAILABLE")
    print(f"Validated threshold:         {VALIDATED_THRESHOLD}")
    print(f"Risk band (LOW/MODERATE/HIGH): {band}")
    print(f"Model-based assessment:      {assessment if score_available else 'UNAVAILABLE'}")
    if tx_class == 3:
        print("\nClass 3 has no ground-truth label in this analysis.")

    if tx_id not in merged_indexed.index:
        print("\n[NOTE] Behavioural feature values are unavailable for this transaction "
              "(not present in the final B feature representation).")
        return result
    rec = merged_indexed.loc[tx_id]

    # ------------------------------------------------------------
    # OBSERVED BEHAVIOURAL CHARACTERISTICS (observations only -- not contributors)
    # ------------------------------------------------------------
    print("\n" + "="*60); print("OBSERVED BEHAVIOURAL CHARACTERISTICS"); print("="*60)
    print("(These are observed values only. A feature listed here is NOT automatically abnormal, unusual,")
    print(" or a contributor to the model's score -- see REFERENCE-BASED UNUSUALNESS and CASE-SPECIFIC")
    print(" MODEL-SENSITIVE CONTRIBUTORS below for those separate, evidence-based claims.)")

    print("\n--- Financial ---")
    fin3, fin4, fin5, fin6 = rec["FIN3"], rec["FIN4"], rec["FIN5"], rec["FIN6"]
    print(f"{display_name('FIN3')}: {fmt_num(fin3)}  -- {interp_share(fin3, 'input')}")
    print(f"{display_name('FIN4')}: {fmt_num(fin4)}  -- {interp_share(fin4, 'output')}")
    print(f"{display_name('FIN5')}: {fmt_num(fin5)}  -- {interp_ratio(fin5, 'BTC value')}")
    print(f"{display_name('FIN6')}: {fmt_num(fin6)}  -- {interp_ratio(fin6, 'transaction size')}")

    print("\n--- Structural ---")
    n_in, n_out, str1, str2 = rec["num_input_addresses"], rec["num_output_addresses"], rec["STR1"], rec["STR2"]
    print(f"{display_name('num_input_addresses')}: {fmt_num(n_in,0)}  -- {interp_count(n_in,'input address','input addresses','aggregates value from')}")
    print(f"{display_name('num_output_addresses')}: {fmt_num(n_out,0)}  -- {interp_count(n_out,'output address','output addresses','distributes value to')}")
    print(f"{display_name('STR1')}: {fmt_num(str1)}  -- {interp_str1(str1)}")
    print(f"{display_name('STR2')}: {'Yes' if str2==1 else 'No'}  -- same-transaction address reuse {'detected' if str2==1 else 'not detected'}.")

    print("\n--- Temporal ---")
    tmp1, tmp2, tmp4, tmp5, tmp6, tmp7, tmp6fa = rec["TMP1"], rec["TMP2"], rec["TMP4"], rec["TMP5"], rec["TMP6"], rec["TMP7"], rec["TMP6_is_first_appearance"]
    print(f"{display_name('TMP1')}: {fmt_num(tmp1)}  -- {interp_age(tmp1,'input')}")
    print(f"{display_name('TMP2')}: {fmt_num(tmp2)}  -- {interp_age(tmp2,'output')}")
    print(f"{display_name('TMP4')}: {fmt_num(tmp4,0)}  -- {interp_prior_tx(tmp4,'sender/input')}")
    print(f"{display_name('TMP5')}: {fmt_num(tmp5,0)}  -- {interp_prior_tx(tmp5,'receiver/output')}")
    print(f"{display_name('TMP6')}: {'N/A' if pd.isna(tmp6) else fmt_num(tmp6)}  -- {interp_tmp6(tmp6)}")
    print(f"{display_name('TMP7')}: {fmt_num(tmp7,0)}  -- {interp_tmp7(tmp7)}")
    print(f"{display_name('TMP6_is_first_appearance')}: {'Yes' if tmp6fa==1 else 'No'}  -- {interp_first_appearance(tmp6fa)}")

    print("\n--- Network ---")
    net1, net2 = rec["NET1"], rec["NET2"]
    print(f"{display_name('NET1')}: {fmt_num(net1,0)}  -- {interp_net(net1,'input')}")
    print(f"{display_name('NET2')}: {fmt_num(net2,0)}  -- {interp_net(net2,'output')}")

    # ============================================================
    # SILENT COMPUTATION: Layer B (reference-based unusualness)
    # Reuses the EXISTING build_case_rows() helper and the EXISTING stored reference
    # statistics (class1_median / class2_median / overall_median / percentile_overall_labelled /
    # unusualness) -- the reference population is not recomputed or redefined here.
    # ============================================================
    case_row_df = build_case_rows([tx_id], {tx_id: score})
    cont_rows = pd.DataFrame()
    bin_rows = pd.DataFrame()
    if len(case_row_df) > 0:
        result["reference_available"] = True
        cont_rows = case_row_df[case_row_df["value_type"] == "continuous"]
        bin_rows = case_row_df[case_row_df["value_type"] == "binary"]

    UNUSUAL_CAP_PER_CATEGORY = 3
    category_unusual = {}
    for cat in ["FINANCIAL", "STRUCTURAL", "TEMPORAL", "NETWORK"]:
        cat_rows = cont_rows[cont_rows["feature_category"] == cat] if result["reference_available"] else pd.DataFrame()
        unusual = cat_rows[cat_rows["unusualness"].isin(["HIGH", "LOW"])].copy()
        if len(unusual) > 0:
            unusual["extremity"] = (unusual["percentile_overall_labelled"] - 50).abs()
            unusual = unusual.sort_values("extremity", ascending=False).head(UNUSUAL_CAP_PER_CATEGORY)
        category_unusual[cat] = unusual.to_dict("records")
    any_unusual_anywhere = any(len(v) > 0 for v in category_unusual.values())

    fp_present = (n_in == 1) and (n_out == 1) and (tmp6fa == 1)

    # ============================================================
    # SILENT COMPUTATION: Layer C (case-specific model-sensitive contributors)
    # Computed LIVE, on demand, for THIS transaction -- same feature-occlusion method as
    # Notebook 07 (train-only baseline, single-feature perturbation, full-graph forward pass),
    # just not limited to a precomputed set. Takes ~7-8 seconds (32 forward passes).
    # ============================================================
    named_cats = ["FINANCIAL", "STRUCTURAL", "TEMPORAL", "NETWORK"]
    key_contributors = []
    attr_this = pd.DataFrame()
    try:
        attr_this = compute_live_attribution(node_idx)
        result["attribution_available"] = True
        attr_named = attr_this.sort_values("absolute_attribution", ascending=False)
        for _, r in attr_named.iterrows():
            delta = float(r["attribution_delta"])
            is_zero = (delta == 0.0) and (abs(r["original_value"] - r["perturbed_value"]) < 1e-9)
            if not is_zero:
                feat = r["feature"]
                key_contributors.append({
                    "feature": feat, "display": display_name(feat), "category": r["feature_category"],
                    "raw_value": rec[feat] if feat in rec.index else None,
                    "original_value": float(r["original_value"]), "perturbed_value": float(r["perturbed_value"]),
                    "delta": delta,
                })
    except Exception as e:
        print(f"[NOTE] Live attribution computation could not be completed for this transaction: {e}")
        result["attribution_available"] = False
    result["n_key_contributors"] = len(key_contributors)
    result["key_contributors"] = [kc["display"] for kc in key_contributors]
    attribution_by_feature = {kc["feature"]: kc for kc in key_contributors}

    CONTRIBUTOR_CAP_PER_CATEGORY = 3
    category_contributors = {cat: [] for cat in named_cats}
    for cat in named_cats:
        cat_kcs = [kc for kc in key_contributors if kc["category"] == cat]
        cat_kcs.sort(key=lambda kc: abs(kc["delta"]), reverse=True)
        category_contributors[cat] = cat_kcs[:CONTRIBUTOR_CAP_PER_CATEGORY]

    # ============================================================
    # PER-FEATURE FORENSIC COMPARISON -- shared renderer used by BOTH the
    # REFERENCE-BASED UNUSUALNESS section and the CASE-SPECIFIC MODEL-SENSITIVE
    # CONTRIBUTORS section below, so a feature's statistical standing and its
    # model sensitivity are always reported from the same underlying numbers.
    # ============================================================
    FEATURE_KIND = {}
    for f_ in ["FIN3", "FIN4", "FIN5", "FIN6", "STR1"]:
        FEATURE_KIND[f_] = "ratio"
    for f_ in ["num_input_addresses", "num_output_addresses", "TMP4", "TMP5", "TMP7",
               "NET1", "NET2", "in_txs_degree", "out_txs_degree"]:
        FEATURE_KIND[f_] = "count"
    for f_ in ["TMP1", "TMP2", "TMP6"]:
        FEATURE_KIND[f_] = "age"
    for f_ in ["total_BTC", "fees", "size", "in_BTC_min", "in_BTC_max", "in_BTC_mean", "in_BTC_median",
               "in_BTC_total", "out_BTC_min", "out_BTC_max", "out_BTC_mean", "out_BTC_median", "out_BTC_total"]:
        FEATURE_KIND[f_] = "monetary"

    def domain_meaning_text(feat, val):
        # Reuses the EXISTING hand-written interpretation helpers / FEATURE_DICTIONARY text only --
        # no new feature meaning is invented here for features whose semantics are not already documented.
        if feat == "FIN3": return interp_share(val, "input")
        if feat == "FIN4": return interp_share(val, "output")
        if feat == "FIN5": return interp_ratio(val, "BTC value")
        if feat == "FIN6": return interp_ratio(val, "transaction size")
        if feat == "num_input_addresses": return interp_count(val, "input address", "input addresses", "aggregates value from")
        if feat == "num_output_addresses": return interp_count(val, "output address", "output addresses", "distributes value to")
        if feat == "STR1": return interp_str1(val)
        if feat == "TMP1": return interp_age(val, "input")
        if feat == "TMP2": return interp_age(val, "output")
        if feat == "TMP4": return interp_prior_tx(val, "sender/input")
        if feat == "TMP5": return interp_prior_tx(val, "receiver/output")
        if feat == "TMP6": return interp_tmp6(val)
        if feat == "TMP7": return interp_tmp7(val)
        if feat == "NET1": return interp_net(val, "input")
        if feat == "NET2": return interp_net(val, "output")
        return None

    def ratio_to_median_text(value, median):
        # Only computed when mathematically meaningful: both values present, finite, same sign,
        # and the denominator is not near zero (a near-zero median would make "Nx the median" unstable).
        if value is None or median is None: return None
        try:
            if pd.isna(value) or pd.isna(median): return None
        except Exception:
            return None
        if abs(median) < 1e-9: return None
        if (value < 0) != (median < 0): return None
        ratio = value / median
        return f"approximately {ratio:.2f}x the overall labelled median"

    def percentile_stat_text(feat, value, pct, ov_med, unusual):
        if pct is None or pd.isna(pct):
            return "Reference percentile is not available for this feature."
        kind = FEATURE_KIND.get(feat, "generic")
        if kind == "count":
            lead = (f"The transaction's observed value is {fmt_num(value,0)}, compared with a labelled-population "
                    f"median of {fmt_num(ov_med,0)} (percentile {pct:.1f}).")
        elif kind == "ratio":
            lead = (f"The observed ratio value {fmt_num(value)} is at the {pct:.1f}th percentile of the labelled "
                    f"reference population (overall median {fmt_num(ov_med)}).")
        elif kind == "age":
            lead = (f"The observed value {fmt_num(value)} time step(s) is at the {pct:.1f}th percentile of the "
                    f"labelled reference population (overall median {fmt_num(ov_med)} time step(s)).")
        else:
            lead = (f"Observed value {fmt_num(value)} is at the {pct:.1f}th percentile of the labelled reference "
                    f"population (overall median {fmt_num(ov_med)}).")
        r = ratio_to_median_text(value, ov_med)
        ratio_sentence = f" The observed value is {r}." if r else ""
        if unusual == "HIGH":
            tail = " This places the transaction in the extreme upper tail (>=95th percentile) of the reference distribution."
        elif unusual == "LOW":
            tail = " This places the transaction in the extreme lower tail (<=5th percentile) of the reference distribution."
        else:
            tail = " This is within the typical (non-extreme) range of the reference distribution."
        return lead + ratio_sentence + tail

    def render_feature_block(feat, indent="    "):
        # Looks up ONLY the already-computed reference statistics for this transaction
        # (cont_rows / bin_rows, built by the existing build_case_rows() helper) -- never
        # recomputes percentiles, medians, or the reference population itself.
        lines = []
        disp = display_name(feat)
        contrib = attribution_by_feature.get(feat)
        is_binary = feat in BINARY_FEATURES

        if is_binary:
            row_df = bin_rows[bin_rows["feature"] == feat] if result["reference_available"] else pd.DataFrame()
            if len(row_df) == 0:
                lines.append(f"{indent}{disp}")
                lines.append(f"{indent}    Reference comparison not available for this feature.")
                return lines
            row = row_df.iloc[0]
            val = row["value"]
            overall_prev = row["reference_prevalence_overall"] * 100
            c1_prev = row["reference_prevalence_class1"] * 100
            c2_prev = row["reference_prevalence_class2"] * 100
            lines.append(f"{indent}{disp}")
            lines.append(f"{indent}    Observed value                           : {'Yes' if val == 1 else 'No'}")
            lines.append(f"{indent}    Reference prevalence (Overall Labelled) : {overall_prev:.2f}%")
            lines.append(f"{indent}    Reference prevalence (Class 1 illicit)  : {c1_prev:.2f}%")
            lines.append(f"{indent}    Reference prevalence (Class 2 licit)    : {c2_prev:.2f}%")
            if feat == "TMP6_is_first_appearance":
                domain = interp_first_appearance(val)
            elif feat == "STR2":
                domain = "Same-transaction address reuse detected." if val == 1 else "No same-transaction address reuse detected."
            else:
                domain = None
            lines.append(f"{indent}    Interpretation                          :")
            if domain:
                lines.append(f"{indent}        {domain}")
            lines.append(f"{indent}        This behaviour occurs in {overall_prev:.2f}% of the overall labelled "
                          f"reference population ({c1_prev:.2f}% of Class 1 illicit, {c2_prev:.2f}% of Class 2 licit).")
        else:
            row_df = cont_rows[cont_rows["feature"] == feat] if result["reference_available"] else pd.DataFrame()
            if len(row_df) == 0:
                lines.append(f"{indent}{disp}")
                lines.append(f"{indent}    Reference comparison not available for this feature.")
                return lines
            row = row_df.iloc[0]
            val, pct, unusual = row["value"], row["percentile_overall_labelled"], row["unusualness"]
            ov_med, c1_med, c2_med = row["overall_median"], row["class1_median"], row["class2_median"]
            lines.append(f"{indent}{disp}")
            lines.append(f"{indent}    Observed value            : {fmt_num(val)}")
            lines.append(f"{indent}    Overall labelled median   : {fmt_num(ov_med)}")
            lines.append(f"{indent}    Class 1 (illicit) median  : {fmt_num(c1_med)}")
            lines.append(f"{indent}    Class 2 (licit) median    : {fmt_num(c2_med)}")
            lines.append(f"{indent}    Percentile (overall)      : {pct:.2f}" if pct is not None and not pd.isna(pct)
                          else f"{indent}    Percentile (overall)      : N/A")
            lines.append(f"{indent}    Unusualness               : {unusual}")
            domain = domain_meaning_text(feat, val)
            stat_text = percentile_stat_text(feat, val, pct, ov_med, unusual)
            class_note = None
            if (unusual in ("HIGH", "LOW") and c1_med is not None and c2_med is not None
                    and not pd.isna(c1_med) and not pd.isna(c2_med) and pct is not None and not pd.isna(pct)):
                above_both = val > c1_med and val > c2_med
                below_both = val < c1_med and val < c2_med
                if above_both or below_both:
                    class_note = ("The observed value is also " + ("above" if above_both else "below") +
                                  " the median values of both labelled classes.")
            lines.append(f"{indent}    Interpretation            :")
            if domain:
                lines.append(f"{indent}        {domain}")
            lines.append(f"{indent}        {stat_text}")
            if class_note:
                lines.append(f"{indent}        {class_note}")

        # Model sensitivity cross-check -- always shown next to the reference standing so the two
        # questions (unusual? / model-sensitive?) are never conflated into one claim.
        if not result["attribution_available"]:
            lines.append(f"{indent}    Model sensitivity         : Case-specific attribution not available for this transaction.")
        elif contrib is not None:
            lines.append(f"{indent}    Model sensitivity         : YES -- attribution {contrib['delta']:+.4f} "
                          f"(case-specific perturbation test).")
            if not is_binary and unusual == "NOT_EXTREME":
                lines.append(f"{indent}        The feature is not extreme relative to the reference population, but "
                              f"case-specific perturbation analysis indicates that the frozen model is sensitive to "
                              f"its value for this case.")
            elif not is_binary and unusual in ("HIGH", "LOW"):
                lines.append(f"{indent}        This feature is both statistically unusual relative to the reference "
                              f"population and model-sensitive for this specific transaction.")
        elif feat in ATTR_ALL:
            lines.append(f"{indent}    Model sensitivity         : NO measurable sensitivity for this transaction "
                          f"(zero attribution under the perturbation test).")
        else:
            lines.append(f"{indent}    Model sensitivity         : N/A (feature not included in the case-specific "
                          f"attribution feature set).")
        return lines

    # ============================================================
    # LAYER B: REFERENCE-BASED UNUSUALNESS
    # ============================================================
    print("\n" + "="*60); print("REFERENCE-BASED UNUSUALNESS"); print("="*60)
    print("Reference populations (Class 3 is NEVER used to define these): Overall Labelled (46,564) | "
          "Class 1 illicit (4,545) | Class 2 licit (42,019)")
    print("Unusualness rule (already implemented, not redefined here): HIGH if percentile >= 95, LOW if")
    print("percentile <= 5, otherwise NOT_EXTREME. A feature is listed below ONLY if it meets this rule --")
    print("an observed value is not called unusual just because it is large or equal to 1.\n")

    if not result["reference_available"]:
        print("Reference-population comparison is not available for this transaction.")
    else:
        for cat in ["FINANCIAL", "STRUCTURAL", "TEMPORAL", "NETWORK"]:
            print(f"{cat}")
            rows = category_unusual[cat]
            if len(rows) == 0:
                print("    No extreme reference-based behaviour detected in this category.")
            else:
                for r in rows:
                    for line in render_feature_block(r["feature"]):
                        print(line)
                    print()
            print()

    if any_unusual_anywhere:
        strongest_cat = max(
            ["FINANCIAL", "STRUCTURAL", "TEMPORAL", "NETWORK"],
            key=lambda c: max((r["percentile_overall_labelled"] - 50) ** 2 for r in category_unusual[c]) if category_unusual[c] else -1
        )
        print(f"{strongest_cat.title()} contains the strongest reference-based unusual characteristics for "
              f"this transaction. (This is a statement about reference-population unusualness only -- see "
              f"CASE-SPECIFIC MODEL-SENSITIVE CONTRIBUTORS below for the separate question of model sensitivity.)")
    else:
        print("No category (Financial, Structural, Temporal, Network) contains a feature meeting the HIGH/LOW "
              "unusualness rule. This transaction's observed behaviour is broadly typical of the labelled "
              "reference population across all four categories.")

    # ============================================================
    # CLASS ASSOCIATION
    # ============================================================
    print("\n" + "="*60); print("CLASS ASSOCIATION"); print("="*60)
    if fp_present:
        c1_pct = fingerprint_df.loc[fingerprint_df["population"]=="Class1_Illicit","percentage"].iloc[0]
        c2_pct = fingerprint_df.loc[fingerprint_df["population"]=="Class2_Licit","percentage"].iloc[0]
        print("Behavioural pattern: 1 input address + 1 output address + first-ever appearance")
        print(f"Class 1 -- labelled illicit prevalence: {c1_pct:.2f}%")
        print(f"Class 2 -- labelled licit prevalence:   {c2_pct:.2f}%")
        print("\nInterpretation: this behavioural fingerprint is substantially more prevalent in the labelled")
        print("Class 1 population than in the labelled Class 2 population. This is supporting evidence, not proof.")
    else:
        print("The 1-input/1-output/first-appearance fingerprint is not present for this transaction; "
              "no class-association statistic from the existing fingerprint analysis applies.")

    # ============================================================
    # LAYER C: CASE-SPECIFIC MODEL-SENSITIVE CONTRIBUTORS
    # ============================================================
    print("\n" + "="*60); print("CASE-SPECIFIC MODEL-SENSITIVE CONTRIBUTORS"); print("="*60)
    print("A feature is called a model-sensitive contributor here ONLY if the existing case-specific")
    print("perturbation/attribution analysis (Notebook 07 methodology, computed live) recorded a non-zero")
    print("output change for this exact transaction. This is never inferred from percentile, correlation,")
    print("raw magnitude, feature family, global importance, or intuition. Unusualness (above) and")
    print("attribution (here) are separate claims about the same feature, shown together below.\n")

    if not result["attribution_available"]:
        print("Case-specific model attribution is not available for this transaction.")
    elif len(key_contributors) == 0:
        print("Case-specific attribution is available for this transaction, but no feature showed a non-zero")
        print("measured contribution under the perturbation test.")
        print("(Zero attribution does not prove a feature is globally irrelevant -- it means this specific")
        print(" transaction's value for that feature already matched the train-only reference baseline.)")
    else:
        for cat in named_cats:
            rows = category_contributors[cat]
            if not rows:
                continue
            print(f"{cat}:")
            for kc in rows:
                for line in render_feature_block(kc["feature"]):
                    print(line)
                print()
            print()
        n_total = len(key_contributors)
        n_shown = sum(len(v) for v in category_contributors.values())
        if n_total > n_shown:
            print(f"({n_total} features total showed non-zero measured sensitivity; the strongest up to "
                  f"{CONTRIBUTOR_CAP_PER_CATEGORY} per category are shown above.)")
        print("Attribution indicates model sensitivity under the perturbation test. It does not establish causality.")

    # ============================================================
    # MODEL ASSESSMENT + WHY THIS TRANSACTION RECEIVED THIS ASSESSMENT
    # ============================================================
    print("\n" + "="*60); print("MODEL ASSESSMENT"); print("="*60)
    print(f"Risk Score: {score:.4f}" if score_available else "Risk Score: UNAVAILABLE")
    print(f"Risk Level: {band}")
    print(f"Validated threshold: {VALIDATED_THRESHOLD}  (0.50 is a presentation boundary only, not separately validated)")

    print("\n" + "="*60); print("WHY THIS TRANSACTION RECEIVED THIS ASSESSMENT"); print("="*60)
    print("Behavioural findings:")
    for cat in ["FINANCIAL", "STRUCTURAL", "TEMPORAL", "NETWORK"]:
        rows = category_unusual.get(cat, [])
        if rows:
            names = ", ".join(display_name(r["feature"]) for r in rows)
            print(f"  - {cat.title()}: {len(rows)} unusual feature(s) -- {names}")
        else:
            print(f"  - {cat.title()}: no extreme behaviour detected")

    print("\nModel-sensitive contributors:")
    if not result["attribution_available"]:
        print("  Case-specific model attribution is not available for this transaction.")
    elif len(key_contributors) == 0:
        print("  Case-specific attribution is available, but no feature showed measurable sensitivity for "
              "this transaction.")
    else:
        names = ", ".join(kc["display"] for kc in key_contributors[:6])
        more = f", and {len(key_contributors)-6} more" if len(key_contributors) > 6 else ""
        print(f"  {names}{more} (see CASE-SPECIFIC MODEL-SENSITIVE CONTRIBUTORS above for full detail).")

    print("\nFORENSIC INTERPRETATION:")
    if band == "HIGH":
        print("The transaction received a high model-based illicit-risk score and exhibits the behavioural")
        print("characteristics listed above that support forensic prioritization. This does not prove illicit")
        print("activity.")
    elif band == "MODERATE":
        print("The transaction shows some characteristics associated with elevated model-based risk, but its")
        print("score remains below the validated high-risk threshold.")
    elif band == "LOW":
        print("The transaction does not exhibit a combination of behavioural characteristics that produces a")
        print("high illicit-risk score under the frozen model.")
    else:
        print("No model-based risk assessment is available for this transaction.")

    # ============================================================
    # LAYER D: FINAL FORENSIC SUMMARY ("WHY THIS TRANSACTION WAS FLAGGED")
    # ============================================================
    print("\n" + "="*60); print("FINAL FORENSIC SUMMARY"); print("="*60)

    def ranked_findings(limit=5):
        flat = [r for cat in named_cats for r in category_unusual[cat]]
        flat.sort(key=lambda r: abs(r["percentile_overall_labelled"] - 50), reverse=True)
        return flat[:limit]

    def findings_block_text(rows):
        if not rows:
            return "No category showed a feature meeting the HIGH/LOW unusualness rule."
        blocks = []
        for idx, r in enumerate(rows, start=1):
            feat = r["feature"]
            word = "high" if r["unusualness"] == "HIGH" else "low"
            blocks.append(
                f"{idx}. {display_name(feat)}\n"
                f"   Observed value : {fmt_num(r['value'])}\n"
                f"   Overall median : {fmt_num(r['overall_median'])}\n"
                f"   Percentile     : {r['percentile_overall_labelled']:.1f}\n"
                f"   Interpretation : unusually {word} relative to the labelled reference population."
            )
        return "\n\n".join(blocks)

    if band == "HIGH":
        findings_block = findings_block_text(ranked_findings(5))
        if key_contributors:
            attributed_text = ", ".join(kc["display"] for kc in key_contributors[:6])
        else:
            attributed_text = "none available" if not result["attribution_available"] else "none with non-zero measured sensitivity"
        summary_text = (
            f"WHY THIS TRANSACTION WAS FLAGGED\n{'-'*50}\n\n"
            f"This transaction received a HIGH model-based illicit-risk score ({score:.6f}).\n\n"
            f"The strongest reference-based behavioural findings were:\n\n"
            f"{findings_block}\n\n"
            f"Case-specific model attribution additionally indicates sensitivity to:\n"
            f"   {attributed_text}\n\n"
            f"These findings support prioritizing the transaction for further forensic investigation.\n"
            f"They do NOT establish illicit activity."
        )
    elif band == "MODERATE":
        findings_block = findings_block_text(ranked_findings(3))
        if key_contributors:
            attributed_text = ", ".join(kc["display"] for kc in key_contributors[:6])
        else:
            attributed_text = "none available" if not result["attribution_available"] else "none with non-zero measured sensitivity"
        summary_text = (
            f"Risk Score: {score:.6f}\nRisk Band: MODERATE\n\n"
            f"Supporting observations (reference-based unusualness):\n\n"
            f"{findings_block}\n\n"
            f"Model-sensitive features (case-specific attribution): {attributed_text}\n\n"
            f"Interpretation: the transaction shows some characteristics associated with elevated model-based "
            f"risk, but its score remains below the validated high-risk threshold ({VALIDATED_THRESHOLD})."
        )
    elif band == "LOW":
        summary_text = (
            f"LOW RISK\nScore: {score:.6f}\n\n"
            f"Reference-based findings:\n" +
            "\n".join(f"  {cat.title()}: {'no extreme features detected' if not category_unusual[cat] else str(len(category_unusual[cat])) + ' unusual feature(s) -- ' + ', '.join(display_name(r['feature']) for r in category_unusual[cat])}"
                      for cat in ["FINANCIAL", "STRUCTURAL", "TEMPORAL", "NETWORK"]) +
            f"\n\nFinal interpretation: the frozen model assigns this transaction a low illicit-risk score. "
            f"No strong combination of reference-based unusual behaviours was identified under the "
            f"implemented analysis."
        )
    else:
        summary_text = f"Transaction {tx_id} has no available model-based illicit-risk score in this analysis."

    print(summary_text)
    result["summary_text"] = summary_text
    return result

print("analyze_transaction() redefined -- ready for use.")

analyze_transaction() redefined -- ready for use.


## Run the analyzer

**Change only `DEMO_TX_ID` below and rerun this cell to analyze any valid transaction ID.**
Everything else (behavioural display, model score, threshold decision, reference comparison,
attribution lookup, forensic summary, presentation panel) updates automatically.

In [29]:
# Change this value to analyze any valid transaction ID.
DEMO_TX_ID = 99943310

result = analyze_transaction(DEMO_TX_ID)

        BITCOIN TRANSACTION FORENSIC ANALYZER
[Analyzer v7 -- full per-feature forensic comparison: observed value, reference percentile,
 class medians, unusualness, and case-specific model sensitivity, shown together per feature]

Transaction ID: 99943310

TRANSACTION OVERVIEW
Transaction ID:              99943310
Dataset class:               Class 3 -- Unknown
Time step:                   42
Model illicit-risk score:    1.000000
Validated threshold:         0.9
Risk band (LOW/MODERATE/HIGH): HIGH
Model-based assessment:      HIGH RISK

Class 3 has no ground-truth label in this analysis.

OBSERVED BEHAVIOURAL CHARACTERISTICS
(These are observed values only. A feature listed here is NOT automatically abnormal, unusual,
 or a contributor to the model's score -- see REFERENCE-BASED UNUSUALNESS and CASE-SPECIFIC
 MODEL-SENSITIVE CONTRIBUTORS below for those separate, evidence-based claims.)

--- Financial ---
Largest Input Share (FIN3): 1.0  -- All recorded input value is concentrated in


REFERENCE-BASED UNUSUALNESS
Reference populations (Class 3 is NEVER used to define these): Overall Labelled (46,564) | Class 1 illicit (4,545) | Class 2 licit (42,019)
Unusualness rule (already implemented, not redefined here): HIGH if percentile >= 95, LOW if
percentile <= 5, otherwise NOT_EXTREME. A feature is listed below ONLY if it meets this rule --
an observed value is not called unusual just because it is large or equal to 1.

FINANCIAL
    Fee Rate (FIN6)
        Observed value            : 0.0
        Overall labelled median   : 0.0
        Class 1 (illicit) median  : 0.0
        Class 2 (licit) median    : 0.0
        Percentile (overall)      : 97.15
        Unusualness               : HIGH
        Interpretation            :
            The fee represents approximately 0.0005% of the transaction's transaction size.
            The observed ratio value 0.0 is at the 97.1th percentile of the labelled reference population (overall median 0.0). The observed value is approximat

## Validation — tested against multiple transactions

Confirms the analyzer works correctly for the three transactions already used elsewhere in this
project's forensic analysis, for one Class 1 and one Class 2 transaction (proving it is not
Class-3-only), and for one deliberately invalid ID (proving graceful, non-crashing error handling).
Displayed values for the three required test cases are cross-checked against
`top20_forensic_case_profiles.csv` to confirm nothing is hardcoded or invented.

In [30]:
print("#"*70); print("# VALIDATION: DEMO_TX_ID = 196098540 (required test case)"); print("#"*70)
r1 = analyze_transaction(196098540)

print("\n\n" + "#"*70); print("# VALIDATION: DEMO_TX_ID = 15340246 (required test case)"); print("#"*70)
r2 = analyze_transaction(15340246)

print("\n\n" + "#"*70); print("# VALIDATION: DEMO_TX_ID = 191501490 (required test case)"); print("#"*70)
r3 = analyze_transaction(191501490)

class1_sample = int(tx_data[tx_data["class"]==1]["txId"].iloc[0])
class2_sample = int(tx_data[tx_data["class"]==2]["txId"].iloc[0])
print("\n\n" + "#"*70); print(f"# VALIDATION: Class 1 sample txId = {class1_sample}"); print("#"*70)
r4 = analyze_transaction(class1_sample)

print("\n\n" + "#"*70); print(f"# VALIDATION: Class 2 sample txId = {class2_sample}"); print("#"*70)
r5 = analyze_transaction(class2_sample)

print("\n\n" + "#"*70); print("# VALIDATION: Invalid txId = 999999999999"); print("#"*70)
r6 = analyze_transaction(999999999999)

print("\n\n" + "="*70); print("CROSS-VALIDATION AGAINST FROZEN OUTPUTS"); print("="*70)
profiles_check = pd.read_csv(os.path.join(OUTPUTS_DIR, "top20_forensic_case_profiles.csv"))
for tx_id, r in [(196098540, r1), (15340246, r2), (191501490, r3)]:
    prow_check = profiles_check[profiles_check["txId"]==tx_id].iloc[0]
    assert r["time_step"] == int(prow_check["Time step"]), f"Time step mismatch for {tx_id}"
    assert abs(r["score"] - float(prow_check["illicit_risk_score"])) < 1e-5, f"Score mismatch for {tx_id}"
    print(f"[PASS] txId {tx_id}: time_step and score match frozen top20_forensic_case_profiles.csv exactly.")

assert r6["found"] == False, "Invalid txId should be reported as not found!"
print("[PASS] Invalid txId correctly reported as not found (no crash).")
assert r4["class"] == 1 and r5["class"] == 2
print("[PASS] Class 1 and Class 2 sample transactions correctly classified and scored via live frozen-model inference.")
print("\nALL CROSS-VALIDATION CHECKS PASSED.")

######################################################################
# VALIDATION: DEMO_TX_ID = 196098540 (required test case)
######################################################################
        BITCOIN TRANSACTION FORENSIC ANALYZER
[Analyzer v7 -- full per-feature forensic comparison: observed value, reference percentile,
 class medians, unusualness, and case-specific model sensitivity, shown together per feature]

Transaction ID: 196098540

TRANSACTION OVERVIEW
Transaction ID:              196098540
Dataset class:               Class 3 -- Unknown
Time step:                   46
Model illicit-risk score:    1.000000
Validated threshold:         0.9
Risk band (LOW/MODERATE/HIGH): HIGH
Model-based assessment:      HIGH RISK

Class 3 has no ground-truth label in this analysis.

OBSERVED BEHAVIOURAL CHARACTERISTICS
(These are observed values only. A feature listed here is NOT automatically abnormal, unusual,
 or a contributor to the model's score -- see REFERENCE-BASED UNUSUA